# House Keeping


In [1]:
import sys
import os
sys.path.append('/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/')

import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats 
from scipy.stats import shapiro , kstest, mannwhitneyu, ttest_rel, wilcoxon
import torch
from torch import nn
from datasets.datasets import MRIDataset, STAREDataset

from helper_func.analyis_helper import clean_df, plot_boxplots, visualize, generate_batch_views, visualize_views, visualize_class_centroids, visualize_feats, prod_feats
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

## Load 

# ENT

In [2]:
tta_ent_lr4_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_lr5_paths =   "/scratch-second/TTA_results/val_ent_only_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_lr6_paths =   "/scratch-second/TTA_results/val_ent_only_decoder_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_only_decoder_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [3]:
tta_ent_lr4 = pd.read_csv(tta_ent_lr4_paths)
tta_ent_lr4 = clean_df(tta_ent_lr4, True, 'brats')
tta_ent_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000
mean,0.343829,0.505521,0.346177,49.841897,45.253520,51.909509,0.354801,0.540272,0.329114,0.476631,0.547562,0.547140
std,0.354227,0.261888,0.267349,26.924194,19.932555,27.071341,0.361736,0.299920,0.283974,0.396388,0.278092,0.333384
min,0.000000,0.000000,0.000000,4.582576,6.708204,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.371953,0.086529,31.039490,30.149628,29.529646,0.000000,0.294622,0.058640,0.000000,0.372001,0.314668
50%,0.140083,0.603527,0.383030,51.070988,47.176239,54.710602,0.246633,0.640323,0.305814,0.532659,0.609189,0.574540
75%,0.732013,0.691751,0.593553,63.163677,56.518559,70.636391,0.733599,0.748318,0.565987,0.837338,0.741027,0.799000
max,0.869421,0.856117,0.749669,111.795128,106.230881,103.975960,0.920353,0.994188,0.867210,1.000000,0.917607,1.000000


In [4]:
tta_ent_lr4_augs = pd.read_csv(tta_ent_lr4_augs_paths)
tta_ent_lr4_augs = clean_df(tta_ent_lr4_augs, True, 'brats')
tta_ent_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.011395,0.246768,0.352390,37.610085,48.263426,38.421830,0.005796,0.247073,0.363825,0.802747,0.743986,0.649359
std,0.017156,0.352812,0.336776,18.248501,27.869697,26.699733,0.008768,0.383039,0.379459,0.377148,0.303046,0.177432
min,0.000869,0.007637,0.006336,14.163321,11.000000,6.708204,0.000435,0.003899,0.003188,0.056166,0.185315,0.390928
25%,0.001764,0.010671,0.060898,27.270649,27.762249,18.250000,0.000892,0.005369,0.032195,0.824138,0.681434,0.543813
50%,0.002118,0.031122,0.320171,35.323845,53.834169,35.640820,0.001060,0.015882,0.304153,0.996101,0.840209,0.660568
75%,0.013836,0.508974,0.639210,47.301480,64.477546,59.228827,0.006978,0.417794,0.667136,1.000000,0.948165,0.772530
max,0.043880,0.738617,0.747630,64.969223,83.815269,72.996582,0.022432,0.893146,0.841788,1.000000,0.985525,0.869346


In [5]:
tta_ent_lr4[tta_ent_lr4.name.isin(tta_ent_lr4_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.533363,0.575143,0.559851,33.657623,46.707603,53.637241,0.508675,0.645084,0.540765,0.794180,0.552880,0.614723
std,0.423868,0.260428,0.155774,17.327098,8.587766,21.033182,0.430829,0.255730,0.216842,0.168733,0.304954,0.175041
min,0.004502,0.124607,0.383030,5.300118,32.249031,36.837482,0.002257,0.232594,0.326858,0.522181,0.085098,0.400694
25%,0.140083,0.615426,0.398676,35.693138,47.979160,36.905285,0.080892,0.640323,0.366857,0.771527,0.493647,0.510943
50%,0.824008,0.645436,0.638536,36.469166,48.425201,50.020973,0.733599,0.692955,0.528833,0.823529,0.592393,0.574540
75%,0.834526,0.700634,0.687846,38.052597,49.648766,56.331608,0.842478,0.727531,0.614069,0.886011,0.675654,0.781772
max,0.863696,0.789613,0.691169,52.773098,55.235859,88.090858,0.884150,0.932018,0.867210,0.967653,0.917607,0.805666


In [6]:
tta_ent_lr5 = pd.read_csv(tta_ent_lr5_paths)
tta_ent_lr5 = clean_df(tta_ent_lr5, True, 'brats')
tta_ent_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.360940,0.558289,0.391973,44.949835,46.782749,39.924534,0.390221,0.687922,0.335536,0.480985,0.489853,0.654005
std,0.305007,0.207776,0.278829,30.439765,19.082402,27.749852,0.320301,0.248733,0.275978,0.363755,0.196948,0.257107
min,0.000000,0.039574,0.000000,5.385165,7.071068,4.000000,0.000000,0.030113,0.000000,0.000000,0.057703,0.000000
25%,0.062176,0.440276,0.115483,15.578706,40.352559,13.594888,0.061197,0.546130,0.063380,0.073078,0.387889,0.536957
50%,0.336539,0.593568,0.441581,44.960335,50.477554,39.973337,0.379668,0.761075,0.298812,0.511571,0.506972,0.709640
75%,0.619635,0.718364,0.662325,66.850636,58.531620,57.345057,0.687998,0.886950,0.589398,0.850148,0.631109,0.831365
max,0.875750,0.849329,0.825148,104.580101,83.026199,99.306847,0.972318,0.989233,0.800369,1.000000,0.857916,1.000000


In [7]:
tta_ent_lr5_augs = pd.read_csv(tta_ent_lr5_augs_paths)
tta_ent_lr5_augs = clean_df(tta_ent_lr5_augs, True, 'brats')
tta_ent_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.232867,0.355814,0.332477,26.642503,35.443951,31.800456,0.182610,0.303307,0.274694,0.762461,0.648565,0.694302
std,0.237212,0.265804,0.280033,24.725406,24.719132,28.420776,0.215678,0.236523,0.260123,0.302463,0.314557,0.269376
min,0.000000,0.000000,0.000000,4.472136,6.164414,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.008175,0.118464,0.063721,12.842329,21.089456,14.531540,0.004105,0.064573,0.033489,0.743238,0.593179,0.628539
50%,0.165699,0.365042,0.332893,23.319148,27.285341,23.024509,0.091414,0.274149,0.227591,0.870412,0.739817,0.764865
75%,0.394243,0.563157,0.597289,26.578265,44.046447,31.218178,0.294274,0.508336,0.465191,0.955391,0.885759,0.848203
max,0.651933,0.794106,0.744984,101.131599,98.254768,107.916634,0.597775,0.706635,0.690703,1.000000,0.933398,0.975429


In [8]:
tta_ent_lr5[tta_ent_lr5.name.isin(tta_ent_lr5_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.470260,0.610120,0.474604,50.504759,51.314151,42.407009,0.432453,0.775096,0.424419,0.598622,0.521009,0.606247
std,0.270748,0.211719,0.248418,30.422662,14.827781,24.572773,0.285881,0.212757,0.260867,0.288740,0.204005,0.239510
min,0.000404,0.039574,0.000931,5.385165,10.770329,5.744563,0.000309,0.030113,0.000477,0.000581,0.057703,0.019778
25%,0.245753,0.555458,0.245111,27.619848,47.903512,21.349951,0.210279,0.745676,0.162833,0.391926,0.431701,0.532621
50%,0.482850,0.655669,0.579070,51.757298,51.587219,44.770504,0.379668,0.858914,0.500856,0.665318,0.538302,0.682496
75%,0.727890,0.762857,0.667147,72.872395,58.274784,56.147715,0.714764,0.892431,0.609945,0.861913,0.678866,0.756781
max,0.859739,0.849329,0.727518,104.580101,75.299400,87.212387,0.885889,0.965970,0.800369,0.951232,0.794314,0.943966


In [9]:
tta_ent_lr6 = pd.read_csv(tta_ent_lr6_paths)
tta_ent_lr6 = clean_df(tta_ent_lr6, True, 'brats')
tta_ent_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.357867,0.527985,0.384007,47.617818,49.358903,37.311655,0.426513,0.772890,0.333659,0.415348,0.416713,0.633729
std,0.291734,0.178890,0.278603,30.328274,18.739636,27.998940,0.307869,0.213451,0.278684,0.348628,0.166477,0.255199
min,0.000000,0.123214,0.000000,5.830952,7.141428,4.000000,0.000000,0.149531,0.000000,0.000000,0.067990,0.000000
25%,0.062852,0.420105,0.088671,21.026131,41.551199,11.140798,0.097376,0.648398,0.048964,0.057742,0.310291,0.490771
50%,0.366589,0.575314,0.395772,46.602699,52.932043,28.964747,0.447398,0.844677,0.333175,0.360259,0.424827,0.690666
75%,0.593169,0.654663,0.645730,69.210155,59.657269,58.514333,0.691939,0.934464,0.555077,0.765882,0.513141,0.830814
max,0.864089,0.785538,0.836796,106.375740,81.030861,99.684502,0.972318,0.989781,0.839993,0.937585,0.794575,1.000000


In [10]:
tta_ent_lr6_augs = pd.read_csv(tta_ent_lr6_augs_paths)
tta_ent_lr6_augs = clean_df(tta_ent_lr6_augs, True, 'brats')
tta_ent_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.249872,0.391262,0.353313,22.986916,28.622195,26.120022,0.217608,0.372320,0.294812,0.703721,0.666163,0.684768
std,0.240114,0.284052,0.303671,20.320980,15.251987,22.490625,0.259270,0.289512,0.279037,0.294473,0.256044,0.280163
min,0.000000,0.000000,0.000000,5.000000,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.020554,0.130524,0.042914,11.776900,18.889696,12.823853,0.010491,0.072289,0.021970,0.523597,0.560851,0.645476
50%,0.270587,0.351322,0.314003,16.309507,27.000000,20.223749,0.162148,0.428979,0.214268,0.816894,0.793148,0.750600
75%,0.467738,0.636547,0.659607,24.859085,32.327191,28.965312,0.323301,0.591974,0.542891,0.917253,0.858066,0.864898
max,0.699393,0.865491,0.782795,78.576073,74.100266,89.582367,0.868949,0.845636,0.780167,1.000000,0.886300,0.970857


In [11]:
tta_ent_lr6[tta_ent_lr6.name.isin(tta_ent_lr6_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.455239,0.576076,0.466956,49.135627,53.272225,34.657547,0.457336,0.838971,0.415662,0.535200,0.449388,0.635257
std,0.268817,0.167426,0.261168,31.778671,15.474610,25.838605,0.279682,0.173937,0.273297,0.305753,0.147875,0.231927
min,0.001121,0.123214,0.005344,7.874008,7.483315,7.000000,0.000928,0.168101,0.002727,0.001414,0.067990,0.051635
25%,0.225863,0.525525,0.220914,14.467716,50.050968,9.218109,0.310386,0.807486,0.137539,0.264140,0.402586,0.526801
50%,0.482418,0.602868,0.573220,52.087421,54.018517,28.653097,0.444634,0.903151,0.471592,0.625670,0.463728,0.687332
75%,0.644001,0.674087,0.660678,75.197559,60.007069,56.280668,0.647256,0.936439,0.621111,0.775585,0.542539,0.810139
max,0.862974,0.785538,0.822606,104.829384,81.030861,81.932899,0.899493,0.981111,0.825118,0.937585,0.661288,0.944957


In [12]:
tta_ent_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_norm_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_norm_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_norm_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_norm_lr5_augs_paths =   "/scratch-second/TTA_results/val_ent_norm_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_norm_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_norm_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [13]:
tta_ent_norm_lr4 = pd.read_csv(tta_ent_norm_lr4_paths)
tta_ent_norm_lr4 = clean_df(tta_ent_norm_lr4, True, 'brats')
tta_ent_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,0.366651,0.506649,0.340152,48.671920,43.521877,54.617075,0.353150,0.534296,0.297004,0.503154,0.554806,0.541488
std,0.348990,0.259985,0.255124,24.141962,18.369936,27.083515,0.349620,0.301915,0.253646,0.396161,0.274686,0.324274
min,0.000000,0.000000,0.000000,4.123106,7.071068,6.082763,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000177,0.358943,0.063467,34.640169,31.354450,32.787218,0.000089,0.258904,0.055807,0.006107,0.416598,0.293684
50%,0.376347,0.598507,0.393078,50.280228,46.398809,55.649786,0.274469,0.625840,0.259310,0.585610,0.626194,0.586939
75%,0.709955,0.707932,0.568468,65.649630,56.793065,72.159307,0.674881,0.760239,0.496948,0.876338,0.745441,0.795397
max,0.868017,0.844469,0.725611,106.513840,77.424156,105.726059,0.964879,0.994116,0.831607,0.994575,0.935673,1.000000


In [14]:
tta_ent_norm_lr4_augs = pd.read_csv(tta_ent_norm_lr4_augs_paths)
tta_ent_norm_lr4_augs = clean_df(tta_ent_norm_lr4_augs, True, 'brats')
tta_ent_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,0.027774,0.188649,0.350123,45.941174,48.721786,43.430784,0.015263,0.129344,0.341101,0.752458,0.573286,0.417284
std,0.047724,0.191805,0.301149,24.106285,18.727145,29.209995,0.026246,0.134428,0.302307,0.274419,0.439844,0.308036
min,0.000131,0.063784,0.003189,20.615528,29.933260,10.099504,0.000066,0.048810,0.001621,0.457375,0.076504,0.096960
25%,0.000220,0.078225,0.253162,34.607969,39.389209,32.862279,0.000110,0.051751,0.221036,0.628688,0.403353,0.270250
50%,0.000309,0.092667,0.503136,48.600410,48.845158,55.625053,0.000155,0.054691,0.440451,0.800000,0.730202,0.443541
75%,0.041595,0.251082,0.523590,58.603996,58.116049,60.096424,0.022862,0.169611,0.510841,0.900000,0.821678,0.577446
max,0.082881,0.409498,0.544044,68.607582,67.386940,64.567795,0.045569,0.284532,0.581231,1.000000,0.913153,0.711351


In [15]:
tta_ent_norm_lr5 = pd.read_csv(tta_ent_norm_lr5_paths)
tta_ent_norm_lr5 = clean_df(tta_ent_norm_lr5, True, 'brats')
tta_ent_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.360996,0.557047,0.391247,45.460544,46.957692,39.807929,0.391873,0.692683,0.333532,0.482226,0.485849,0.661390
std,0.306765,0.208111,0.278975,30.135162,18.827044,27.692194,0.323027,0.250989,0.275365,0.366324,0.196453,0.254386
min,0.000000,0.033059,0.000000,5.099020,7.141428,4.000000,0.000000,0.023678,0.000000,0.000000,0.054749,0.000000
25%,0.062679,0.430509,0.104920,16.448488,40.511770,13.621351,0.072720,0.551814,0.057661,0.090380,0.394672,0.551547
50%,0.350673,0.596434,0.401667,45.487017,50.134663,37.982346,0.371800,0.762773,0.269172,0.519582,0.497955,0.711593
75%,0.635335,0.714701,0.652603,67.202008,58.545691,58.361793,0.692753,0.892824,0.579150,0.854459,0.621516,0.833335
max,0.878459,0.848177,0.823821,104.361870,83.372948,99.266312,0.973472,0.990575,0.793562,1.000000,0.854372,1.000000


In [16]:
tta_ent_norm_lr5_augs = pd.read_csv(tta_ent_norm_lr5_augs_paths)
tta_ent_norm_lr5_augs = clean_df(tta_ent_norm_lr5_augs, True, 'brats')
tta_ent_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.241213,0.374813,0.349764,25.984379,35.325936,32.075659,0.191018,0.319221,0.289871,0.742124,0.634128,0.689569
std,0.238168,0.256057,0.278743,26.243064,25.367886,29.193979,0.220196,0.227073,0.260082,0.303833,0.314307,0.276218
min,0.000000,0.000000,0.000000,4.472136,6.633250,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.017041,0.159882,0.084136,12.942050,19.528120,13.605737,0.008598,0.100793,0.044668,0.724003,0.565203,0.617118
50%,0.172840,0.388787,0.409943,18.042999,27.092434,23.000000,0.095400,0.294773,0.312772,0.868250,0.712940,0.767691
75%,0.418900,0.568526,0.601096,25.786712,45.189035,33.118283,0.347666,0.512979,0.477915,0.929166,0.874802,0.856316
max,0.656900,0.775728,0.745226,105.777122,98.198776,107.916634,0.594712,0.676929,0.686286,1.000000,0.920953,0.974344


In [17]:
tta_ent_norm_lr6 = pd.read_csv(tta_ent_norm_lr6_paths)
tta_ent_norm_lr6 = clean_df(tta_ent_norm_lr6, True, 'brats')
tta_ent_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.358273,0.527867,0.384075,47.589086,49.357246,37.275013,0.426640,0.773152,0.333393,0.415880,0.416478,0.635001
std,0.292207,0.178770,0.278823,30.349940,18.737871,28.002980,0.307969,0.213610,0.278524,0.348961,0.166410,0.255281
min,0.000000,0.123165,0.000000,5.830952,7.141428,4.000000,0.000000,0.150029,0.000000,0.000000,0.067953,0.000000
25%,0.064598,0.419570,0.088199,20.967446,41.529080,10.885790,0.096920,0.652029,0.048543,0.061971,0.309388,0.492417
50%,0.364963,0.574804,0.398277,46.644827,53.019714,28.904643,0.443625,0.844843,0.333934,0.360934,0.424576,0.690415
75%,0.596135,0.653904,0.645530,69.212778,59.642551,58.429831,0.690271,0.934952,0.553719,0.766243,0.512885,0.832261
max,0.864660,0.785357,0.836811,106.323563,81.074043,99.676979,0.972318,0.989876,0.840174,0.938464,0.794648,1.000000


In [18]:
tta_ent_norm_lr6_augs = pd.read_csv(tta_ent_norm_lr6_augs_paths)
tta_ent_norm_lr6_augs = clean_df(tta_ent_norm_lr6_augs, True, 'brats')
tta_ent_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.249659,0.391998,0.355338,22.957811,28.630911,26.088353,0.217662,0.373117,0.298007,0.701678,0.665559,0.683550
std,0.240475,0.283984,0.306042,20.330880,15.258945,22.525895,0.260102,0.289380,0.283307,0.299937,0.255201,0.282185
min,0.000000,0.000000,0.000000,5.000000,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.021623,0.130570,0.044277,11.736210,18.889696,12.383371,0.011038,0.072373,0.022689,0.521589,0.558480,0.643869
50%,0.271890,0.352504,0.314329,16.278820,27.000000,20.223749,0.162981,0.429255,0.214562,0.819515,0.789315,0.750000
75%,0.466735,0.636999,0.659663,24.654540,32.327191,28.965312,0.322804,0.591494,0.541030,0.915861,0.856639,0.863479
max,0.700928,0.865330,0.798682,78.612022,74.100266,89.582367,0.871901,0.845418,0.779742,1.000000,0.886202,0.971143


In [19]:
tta_ent_first_layer_lr4_paths =    "/scratch-second/TTA_results/val_ent_first_layer_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_first_layer_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_first_layer_lr5_augs_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_first_layer_lr6_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_first_layer_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [20]:
tta_ent_first_layer_lr4 = pd.read_csv(tta_ent_first_layer_lr4_paths)
tta_ent_first_layer_lr4 = clean_df(tta_ent_first_layer_lr4, True, 'brats')
tta_ent_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000
mean,0.359067,0.500542,0.343747,41.718270,39.516525,49.714906,0.355345,0.543401,0.289259,0.574243,0.540555,0.583841
std,0.342438,0.258216,0.245027,26.959754,18.273791,27.557109,0.356334,0.309654,0.242647,0.397830,0.269009,0.306123
min,0.000000,0.000000,0.000000,3.000000,6.403124,6.708204,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.029087,0.348708,0.091260,19.824228,26.019224,21.661016,0.014758,0.235935,0.049956,0.084677,0.402969,0.416586
50%,0.212230,0.590891,0.338877,40.666939,40.607880,50.914116,0.173666,0.660753,0.218934,0.700441,0.613361,0.618035
75%,0.726607,0.684938,0.568749,54.380600,54.691818,66.128662,0.725206,0.771633,0.502533,0.906494,0.726224,0.813644
max,0.861177,0.849894,0.736908,108.374352,70.007141,105.254913,0.968293,0.995545,0.855863,1.000000,0.927582,1.000000


In [21]:
tta_ent_first_layer_lr4_augs = pd.read_csv(tta_ent_first_layer_lr4_augs_paths)
tta_ent_first_layer_lr4_augs = clean_df(tta_ent_first_layer_lr4_augs, True, 'brats')
tta_ent_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.182918,0.193878,0.343280,32.666915,51.949390,45.744707,0.136925,0.124748,0.298965,0.916702,0.682442,0.582748
std,0.289874,0.184880,0.247102,22.141527,20.942612,28.371400,0.248546,0.126188,0.249664,0.088969,0.382974,0.186683
min,0.000115,0.041958,0.004664,5.385165,29.698484,10.000000,0.000057,0.022193,0.002349,0.777778,0.054838,0.316667
25%,0.016115,0.069301,0.180700,21.686343,35.166424,27.653757,0.008364,0.054610,0.106100,0.862165,0.476168,0.450925
50%,0.075694,0.123492,0.362021,27.500000,50.497921,43.272405,0.039517,0.065982,0.280833,0.950704,0.845903,0.629117
75%,0.163139,0.257720,0.500693,42.243750,63.334247,62.541997,0.089564,0.158004,0.529543,0.978958,0.967350,0.708147
max,0.757463,0.520627,0.663795,68.709534,83.333069,86.510696,0.638231,0.354834,0.569471,1.000000,0.987256,0.795568


In [22]:
tta_ent_first_layer_lr5 = pd.read_csv(tta_ent_first_layer_lr5_paths)
tta_ent_first_layer_lr5 = clean_df(tta_ent_first_layer_lr5, True, 'brats')
tta_ent_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.366988,0.557658,0.389673,43.507005,45.169459,40.274349,0.392407,0.690458,0.330216,0.498395,0.488481,0.669258
std,0.309358,0.211002,0.278475,28.707755,19.411618,28.302361,0.324010,0.256064,0.273897,0.369661,0.199009,0.255041
min,0.000000,0.029839,0.000000,4.566020,7.141428,4.000000,0.000000,0.021561,0.000000,0.000000,0.048432,0.000000
25%,0.067250,0.427967,0.111931,16.489477,38.705249,13.740342,0.068660,0.546308,0.062316,0.093616,0.399666,0.568590
50%,0.369407,0.599373,0.385616,45.133572,49.606535,36.347513,0.355696,0.774624,0.261362,0.540281,0.497731,0.716001
75%,0.665237,0.724516,0.651841,63.518821,57.459989,60.302164,0.685200,0.895730,0.574261,0.862650,0.626744,0.841151
max,0.879805,0.863925,0.823428,103.865295,87.584244,99.153160,0.974625,0.992393,0.779422,1.000000,0.855780,1.000000


In [23]:
tta_ent_first_layer_lr5_augs = pd.read_csv(tta_ent_first_layer_lr5_augs_paths)
tta_ent_first_layer_lr5_augs = clean_df(tta_ent_first_layer_lr5_augs, True, 'brats')
tta_ent_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.234080,0.371081,0.344435,25.553708,34.578469,31.665833,0.184255,0.315050,0.288820,0.765612,0.645476,0.698636
std,0.241947,0.262945,0.279956,23.595406,25.425006,28.438993,0.218303,0.228720,0.271152,0.302250,0.312268,0.272391
min,0.000000,0.000000,0.000000,4.358899,7.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.013717,0.120038,0.076757,12.951562,15.652155,14.178078,0.006910,0.064591,0.040923,0.782599,0.574844,0.612023
50%,0.144524,0.400682,0.325213,19.230379,27.650971,23.486439,0.078780,0.338515,0.223709,0.868816,0.729649,0.792686
75%,0.414605,0.575420,0.598148,26.544501,44.126227,31.359303,0.302045,0.517374,0.463284,0.951548,0.883681,0.852326
max,0.673702,0.765908,0.779373,96.569145,97.867256,107.790306,0.572019,0.657971,0.778252,1.000000,0.925121,0.977027


In [24]:
tta_ent_first_layer_lr6 = pd.read_csv(tta_ent_first_layer_lr6_paths)
tta_ent_first_layer_lr6 = clean_df(tta_ent_first_layer_lr6, True, 'brats')
tta_ent_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.359313,0.528865,0.384188,47.559192,48.940632,37.223095,0.427020,0.772982,0.333095,0.417395,0.417702,0.636496
std,0.292529,0.179259,0.278854,30.357859,19.397086,27.985866,0.308070,0.213816,0.278223,0.348535,0.167332,0.255389
min,0.000000,0.123182,0.000000,5.830952,7.141428,4.000000,0.000000,0.149320,0.000000,0.000000,0.067962,0.000000
25%,0.064070,0.420334,0.088861,20.947831,41.498818,10.688188,0.096440,0.654504,0.048732,0.068889,0.310340,0.496250
50%,0.366087,0.574488,0.400183,46.560829,52.916752,28.856262,0.448100,0.844817,0.332163,0.363805,0.424713,0.692123
75%,0.602337,0.654002,0.645516,69.327877,59.602658,58.441783,0.689716,0.934974,0.554624,0.766572,0.513062,0.832696
max,0.865528,0.809794,0.836789,106.339554,81.049370,99.610245,0.972318,0.990066,0.839266,0.939447,0.795363,1.000000


In [25]:
tta_ent_first_layer_lr6_augs = pd.read_csv(tta_ent_first_layer_lr6_augs_paths)
tta_ent_first_layer_lr6_augs = clean_df(tta_ent_first_layer_lr6_augs, True, 'brats')
tta_ent_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.252308,0.396568,0.355436,22.830257,28.483204,26.028989,0.219704,0.376181,0.296395,0.705767,0.666173,0.684641
std,0.241902,0.282833,0.302895,20.240511,15.278419,22.495111,0.260725,0.287111,0.278675,0.294516,0.255954,0.282573
min,0.000000,0.000000,0.000000,4.690416,5.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.021500,0.130466,0.051376,11.736210,18.864313,12.396630,0.010978,0.072294,0.026562,0.525988,0.559697,0.643445
50%,0.294158,0.367454,0.315668,16.278820,26.720778,20.223749,0.177695,0.428962,0.215984,0.836785,0.789358,0.748598
75%,0.465390,0.636352,0.666523,24.671226,32.337061,28.760402,0.323254,0.591926,0.538918,0.914314,0.859737,0.866329
max,0.707233,0.865336,0.782677,78.523880,74.100266,89.582367,0.873672,0.845174,0.777370,1.000000,0.886483,0.971904


### Commons L-4


In [26]:
common_augs = tta_ent_lr5_augs[
    tta_ent_lr5_augs['name'].isin(tta_ent_first_layer_lr5_augs['name']) &
    tta_ent_lr5_augs['name'].isin(tta_ent_norm_lr5_augs['name'])
]
common_augs

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00007-000.npy,0.651933,0.662061,0.603677,13.000000,24.000000,18.000000,0.517911,0.535991,0.500432,0.879534,0.865674,0.760596
1,BraTS-SSA-00010-000.npy,0.022012,0.214300,0.072094,45.088802,16.000000,18.493242,0.011128,0.222888,0.038085,1.000000,0.206349,0.673524
2,BraTS-SSA-00011-000.npy,0.232066,0.224486,0.089839,24.166092,27.184555,29.017237,0.132398,0.128307,0.047643,0.938715,0.896483,0.785922
3,BraTS-SSA-00015-000.npy,0.606119,0.649997,0.014264,26.000000,31.032242,34.423828,0.467588,0.521951,0.007199,0.861291,0.861291,0.763848
4,BraTS-SSA-00044-000.npy,0.006477,0.085221,0.119948,34.263683,79.392700,88.628448,0.003249,0.044649,0.064571,1.000000,0.933398,0.842362
5,BraTS-SSA-00051-000.npy,0.223547,0.748881,0.736646,6.275128,10.000000,15.132746,0.127768,0.635296,0.656938,0.892857,0.911925,0.838366
6,BraTS-SSA-00055-000.npy,0.497884,0.405735,0.271652,8.817591,27.000000,21.876890,0.555490,0.262441,0.158160,0.451103,0.893701,0.961871
7,BraTS-SSA-00056-000.npy,0.166316,0.506586,0.433985,13.638182,12.083046,12.727922,0.092411,0.540754,0.335216,0.830455,0.476480,0.615267
8,BraTS-SSA-00057-000.npy,0.359697,0.324348,0.595160,25.632011,42.941822,24.000000,0.236502,0.206866,0.453444,0.750779,0.750657,0.865726
9,BraTS-SSA-00095-000.npy,0.000000,0.000000,0.000000,82.425728,77.323021,75.901253,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
mask = tta_ent_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.244903,0.372750,0.348444,26.861993,35.540187,31.998601,0.192111,0.318358,0.288371,0.749959,0.636220,0.689445
std,0.237354,0.261765,0.278196,25.382914,25.392644,29.185375,0.217245,0.232957,0.259757,0.305396,0.318160,0.275856
min,0.000000,0.000000,0.000000,4.472136,6.164414,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.015376,0.162440,0.080966,12.684659,19.507344,13.930334,0.007759,0.099761,0.042864,0.735696,0.554279,0.624115
50%,0.166316,0.405735,0.394134,24.166092,27.184555,22.090721,0.092411,0.285857,0.297023,0.861291,0.728977,0.763848
75%,0.428790,0.572291,0.599418,26.768091,45.151072,32.286728,0.352045,0.512874,0.476938,0.943164,0.879688,0.854044
max,0.651933,0.794106,0.744984,101.131599,98.254768,107.916634,0.597775,0.706635,0.690703,1.000000,0.933398,0.975429


In [28]:
mask = tta_ent_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.246139,0.388813,0.360985,25.718943,34.628370,31.856893,0.193822,0.330715,0.303217,0.753276,0.632872,0.693532
std,0.242322,0.257572,0.277394,24.230084,26.120704,29.205097,0.219935,0.223694,0.270615,0.305316,0.315555,0.278871
min,0.000000,0.000000,0.000000,4.358899,7.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.023510,0.154397,0.105424,12.903124,15.148815,13.193709,0.011919,0.132796,0.056635,0.779408,0.552157,0.611267
50%,0.144882,0.408519,0.392821,18.138357,27.000000,23.000000,0.079196,0.406660,0.294652,0.867676,0.706918,0.789759
75%,0.434814,0.588226,0.603020,27.089003,45.298990,32.482039,0.355400,0.519121,0.475715,0.944862,0.875412,0.852472
max,0.673702,0.765908,0.779373,96.569145,97.867256,107.790306,0.572019,0.657971,0.778252,1.000000,0.925121,0.977027


In [29]:
mask = tta_ent_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.241213,0.374813,0.349764,25.984379,35.325936,32.075659,0.191018,0.319221,0.289871,0.742124,0.634128,0.689569
std,0.238168,0.256057,0.278743,26.243064,25.367886,29.193979,0.220196,0.227073,0.260082,0.303833,0.314307,0.276218
min,0.000000,0.000000,0.000000,4.472136,6.633250,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.017041,0.159882,0.084136,12.942050,19.528120,13.605737,0.008598,0.100793,0.044668,0.724003,0.565203,0.617118
50%,0.172840,0.388787,0.409943,18.042999,27.092434,23.000000,0.095400,0.294773,0.312772,0.868250,0.712940,0.767691
75%,0.418900,0.568526,0.601096,25.786712,45.189035,33.118283,0.347666,0.512979,0.477915,0.929166,0.874802,0.856316
max,0.656900,0.775728,0.745226,105.777122,98.198776,107.916634,0.594712,0.676929,0.686286,1.000000,0.920953,0.974344


In [30]:
mask = tta_ent_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.264771,0.419096,0.334078,23.977264,28.219795,28.138810,0.233418,0.409999,0.272560,0.693514,0.651477,0.679538
std,0.221506,0.291510,0.292987,22.033959,16.810826,23.950223,0.260077,0.296476,0.260615,0.268043,0.266588,0.299056
min,0.000000,0.000000,0.000000,5.830952,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.035420,0.140163,0.042914,11.776900,18.233092,13.746524,0.018184,0.076504,0.021970,0.523597,0.560851,0.645476
50%,0.299724,0.555661,0.314003,13.341664,23.021729,20.223749,0.181853,0.500754,0.214268,0.780488,0.763917,0.750600
75%,0.467738,0.659694,0.627325,26.009996,32.951735,29.120394,0.323301,0.637708,0.516916,0.889790,0.839803,0.880107
max,0.632931,0.865491,0.733706,78.576073,74.100266,89.582367,0.868949,0.845636,0.655876,1.000000,0.886300,0.970857


In [31]:
mask = tta_ent_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.267426,0.425476,0.336632,23.800095,28.057767,28.035092,0.235423,0.414700,0.274649,0.696226,0.651200,0.679171
std,0.222665,0.289405,0.292177,21.937287,16.836917,23.958284,0.260835,0.292946,0.260682,0.268125,0.266723,0.301611
min,0.000000,0.000000,0.000000,5.761815,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.035516,0.140174,0.051376,11.736210,18.246621,13.598289,0.018236,0.076528,0.026562,0.525988,0.559697,0.643445
50%,0.316304,0.556737,0.315668,13.304134,23.021729,20.223749,0.202000,0.499834,0.215984,0.776000,0.747389,0.745875
75%,0.465390,0.659262,0.634532,25.872851,32.961605,29.051168,0.323254,0.636616,0.522368,0.890811,0.839272,0.883908
max,0.636432,0.865336,0.733985,78.523880,74.100266,89.582367,0.873672,0.845174,0.657009,1.000000,0.886483,0.971904


In [32]:
mask = tta_ent_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.264618,0.419309,0.334511,23.938706,28.236570,28.100475,0.233454,0.410237,0.273047,0.694021,0.650397,0.678144
std,0.221606,0.291578,0.292939,22.046815,16.819918,23.994101,0.260835,0.296561,0.260735,0.268024,0.266017,0.301383
min,0.000000,0.000000,0.000000,5.830952,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.035547,0.140236,0.044277,11.736210,18.233092,13.541675,0.018253,0.076565,0.022689,0.521589,0.558480,0.643869
50%,0.294580,0.556551,0.314329,13.341664,23.021729,20.223749,0.178115,0.500533,0.214562,0.776892,0.754728,0.745882
75%,0.466735,0.659773,0.627499,25.856166,32.951735,29.146081,0.322804,0.636980,0.515614,0.891007,0.839012,0.878928
max,0.635405,0.865330,0.733829,78.612022,74.100266,89.582367,0.871901,0.845418,0.656584,1.000000,0.886202,0.971143


In [33]:
common_rows = tta_ent_lr4[
    tta_ent_lr4['name'].isin(tta_ent_first_layer_lr4['name']) &
    tta_ent_lr4['name'].isin(tta_ent_norm_lr4['name'])
]
common_rows

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00007-000.npy,0.792694,0.776874,0.749669,39.824615,56.518559,46.786751,0.920353,0.748318,0.740512,0.696136,0.807695,0.759054
1,BraTS-SSA-00008-000.npy,0.585536,0.529937,0.010324,30.675724,63.051567,22.580952,0.598665,0.640427,0.005189,0.572970,0.451961,0.985401
2,BraTS-SSA-00010-000.npy,0.129163,0.397981,0.000564,50.219517,30.083218,52.193863,0.078092,0.537193,0.000324,0.373276,0.316072,0.002151
3,BraTS-SSA-00011-000.npy,0.810440,0.691751,0.127516,63.163677,60.745369,62.871693,0.785260,0.800201,0.071834,0.837288,0.609189,0.567094
4,BraTS-SSA-00014-000.npy,0.304930,0.743770,0.272162,105.564438,75.802376,100.626038,0.320860,0.777083,0.239773,0.290507,0.713196,0.314668
5,BraTS-SSA-00015-000.npy,0.863696,0.615426,0.398676,52.773098,55.235859,50.020973,0.842478,0.640323,0.326858,0.886011,0.592393,0.510943
6,BraTS-SSA-00025-000.npy,0.666384,0.478782,0.185905,31.591139,22.293497,24.166092,0.889762,0.671546,0.102586,0.532659,0.372001,0.989829
9,BraTS-SSA-00037-000.npy,0.002298,0.796793,0.486454,26.019224,6.708204,5.000000,0.019737,0.717894,0.349672,0.001220,0.895176,0.799000
10,BraTS-SSA-00041-000.npy,0.002284,0.655819,0.110744,19.261360,21.118711,16.093477,0.001144,0.593163,0.058640,1.000000,0.733276,0.993407
11,BraTS-SSA-00044-000.npy,0.834526,0.789613,0.687846,5.300118,49.648766,88.090858,0.733599,0.692955,0.614069,0.967653,0.917607,0.781772


In [34]:
mask = tta_ent_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.412758,0.547928,0.361213,44.722553,44.002879,51.084704,0.400905,0.598135,0.343320,0.580020,0.578666,0.595943
std,0.350341,0.217847,0.257273,27.089790,17.394322,27.490110,0.356576,0.263991,0.290689,0.362084,0.243973,0.288182
min,0.000000,0.002297,0.000564,4.582576,6.708204,5.000000,0.000000,0.001762,0.000324,0.000000,0.003298,0.001358
25%,0.055413,0.475576,0.119856,30.417084,32.249031,27.924004,0.028655,0.523488,0.063749,0.290507,0.451961,0.465587
50%,0.426984,0.631135,0.391122,39.824615,45.923851,52.193863,0.323210,0.655580,0.305814,0.771527,0.609189,0.640227
75%,0.775372,0.691751,0.593553,59.709717,55.235859,72.449982,0.768431,0.777083,0.614069,0.867712,0.741027,0.781772
max,0.869421,0.796793,0.749669,111.795128,75.802376,103.975960,0.920353,0.994188,0.867210,1.000000,0.917607,1.000000


In [35]:
mask = tta_ent_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.407585,0.532560,0.347806,39.519491,40.771063,48.799485,0.403720,0.582200,0.292993,0.618964,0.572276,0.603213
std,0.337459,0.223956,0.239903,25.070370,18.650964,29.010767,0.353712,0.277488,0.242898,0.370142,0.241148,0.300088
min,0.000000,0.001461,0.000000,3.000000,6.403124,6.708204,0.000000,0.001264,0.000000,0.000000,0.001730,0.000000
25%,0.049456,0.451711,0.154857,19.824228,26.019224,20.216297,0.072249,0.441695,0.084068,0.316393,0.459501,0.459442
50%,0.365660,0.605571,0.338877,40.655254,46.400433,49.132473,0.425120,0.673080,0.218934,0.817012,0.615756,0.618035
75%,0.784329,0.684938,0.561351,53.131748,54.713799,66.128662,0.758105,0.771633,0.502533,0.906494,0.726224,0.813644
max,0.861177,0.782074,0.736908,102.198090,70.007141,105.254913,0.968293,0.995545,0.855863,1.000000,0.927582,1.000000


In [36]:
mask = tta_ent_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.429399,0.541474,0.358635,45.410420,43.510164,53.122705,0.415458,0.583247,0.318391,0.587943,0.585337,0.579151
std,0.337946,0.217928,0.246660,25.430851,18.951818,29.511501,0.344350,0.267250,0.258160,0.369022,0.236086,0.297329
min,0.000000,0.003559,0.000000,4.123106,7.071068,6.082763,0.000000,0.002854,0.000000,0.000000,0.004727,0.000000
25%,0.033994,0.487004,0.183549,31.400637,31.025002,28.425341,0.020007,0.492268,0.101323,0.330253,0.486610,0.457265
50%,0.460912,0.629214,0.399631,41.475178,46.786751,55.331722,0.451757,0.648115,0.266958,0.746267,0.631103,0.643581
75%,0.714448,0.705559,0.565476,58.000858,56.964901,73.786125,0.740306,0.746011,0.562453,0.882303,0.740461,0.795081
max,0.868017,0.783723,0.725611,106.513840,77.424156,105.726059,0.964879,0.994116,0.831607,0.994575,0.935673,1.000000


### L5

In [37]:
mask = tta_ent_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.431980,0.598617,0.413149,47.318989,46.624324,38.335966,0.446482,0.750029,0.362009,0.497163,0.522721,0.653347
std,0.321054,0.179258,0.265154,30.285590,19.823319,29.091575,0.316066,0.213106,0.275372,0.364378,0.181854,0.229107
min,0.000000,0.084133,0.000931,5.385165,7.071068,4.000000,0.000000,0.072726,0.000477,0.000000,0.075772,0.019778
25%,0.127364,0.567618,0.195888,26.000000,43.416588,11.401754,0.111320,0.705656,0.116717,0.183918,0.437183,0.532704
50%,0.412503,0.655354,0.454844,46.549381,50.944096,37.054016,0.484748,0.800961,0.319354,0.542714,0.534136,0.702062
75%,0.757607,0.716649,0.656925,67.753960,61.351448,54.827000,0.769583,0.887537,0.595363,0.854333,0.615994,0.786139
max,0.875750,0.797624,0.825148,103.963318,76.896034,99.306847,0.898268,0.989233,0.800369,0.951232,0.857916,1.000000


In [38]:
mask = tta_ent_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.446678,0.600620,0.409228,46.595739,46.061155,39.366220,0.457858,0.757952,0.353058,0.514039,0.522418,0.666091
std,0.322552,0.179053,0.264361,30.138027,20.133234,29.855999,0.319700,0.213654,0.272578,0.365956,0.182501,0.232304
min,0.000000,0.095021,0.000484,4.566020,7.141428,4.000000,0.000000,0.084754,0.000248,0.000000,0.075593,0.010252
25%,0.139324,0.565754,0.176437,25.961510,43.840622,11.657596,0.182654,0.731593,0.101149,0.164857,0.446207,0.554303
50%,0.423826,0.652206,0.387395,46.987240,51.352222,33.786091,0.513144,0.807550,0.261805,0.560424,0.531456,0.715864
75%,0.783959,0.727074,0.646717,65.211189,60.450783,55.901699,0.776265,0.896946,0.574288,0.864243,0.617329,0.809399
max,0.879805,0.799665,0.823428,103.865295,77.077881,99.153160,0.902887,0.992393,0.779422,0.957453,0.855780,1.000000


In [39]:
mask = tta_ent_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.436555,0.598606,0.412122,47.164007,46.588070,38.236037,0.453833,0.756267,0.358354,0.502541,0.519799,0.658702
std,0.321185,0.177710,0.263228,30.312257,19.613857,28.879740,0.318771,0.212330,0.273333,0.363882,0.180354,0.229399
min,0.000000,0.094597,0.000892,5.099020,7.141428,4.000000,0.000000,0.083988,0.000458,0.000000,0.075579,0.017379
25%,0.162980,0.562509,0.199542,26.038433,43.737854,11.571497,0.126402,0.723129,0.117938,0.173472,0.446025,0.541308
50%,0.391171,0.644128,0.403172,47.431530,50.941143,33.949947,0.493866,0.804073,0.271602,0.563473,0.528278,0.704939
75%,0.772014,0.713522,0.647320,69.756706,60.475201,55.803665,0.774085,0.893322,0.580807,0.857661,0.613054,0.789183
max,0.878459,0.797058,0.823821,103.660469,77.362785,99.266312,0.898373,0.990575,0.793562,0.955684,0.854372,1.000000


### L6

In [41]:
mask = tta_ent_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.405095,0.565217,0.404179,49.668909,48.576888,36.256080,0.450507,0.809812,0.365697,0.449061,0.451725,0.623389
std,0.314857,0.158338,0.266039,29.287858,19.690356,29.575630,0.313069,0.186293,0.282466,0.362524,0.157473,0.219819
min,0.000000,0.123214,0.005344,7.615773,7.141428,4.000000,0.000000,0.149531,0.002727,0.000000,0.067990,0.051635
25%,0.077697,0.545491,0.134858,26.702061,43.428101,9.380832,0.194107,0.766673,0.079176,0.084656,0.388660,0.499755
50%,0.389932,0.590910,0.458899,46.794193,52.096066,25.961510,0.494217,0.880230,0.357297,0.400708,0.433047,0.683706
75%,0.647255,0.664216,0.653478,69.939957,59.715996,57.825161,0.704367,0.936051,0.601313,0.783957,0.536237,0.741901
max,0.864089,0.780375,0.801733,106.375740,77.221756,99.684502,0.899493,0.989781,0.839993,0.937585,0.794575,1.000000


In [173]:
mask = tta_ent_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.407214,0.565373,0.404211,49.582099,48.543292,36.178283,0.452243,0.810400,0.364618,0.451180,0.451786,0.625782
std,0.315646,0.158419,0.266205,29.338863,19.700763,29.584786,0.313415,0.186309,0.281773,0.361763,0.157710,0.220202
min,0.000000,0.123182,0.004973,7.549834,7.141428,4.000000,0.000000,0.149320,0.002536,0.000000,0.067962,0.051132
25%,0.077704,0.544523,0.135894,26.645824,43.428101,9.273619,0.197504,0.767353,0.079846,0.106383,0.388438,0.500790
50%,0.392608,0.591722,0.460157,46.758957,51.972572,26.034569,0.492682,0.880852,0.357794,0.404519,0.433307,0.685302
75%,0.646378,0.664158,0.653153,70.153046,59.690872,57.982758,0.707366,0.936606,0.602188,0.790833,0.535291,0.741779
max,0.865528,0.781107,0.802570,106.339554,77.188080,99.610245,0.899070,0.990066,0.839266,0.939447,0.795363,1.000000


In [42]:
mask = tta_ent_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.405763,0.565085,0.404206,49.632554,48.567422,36.222453,0.451302,0.810232,0.365177,0.449475,0.451491,0.624683
std,0.315335,0.158342,0.266068,29.330849,19.686290,29.580564,0.313277,0.186216,0.282128,0.362456,0.157633,0.220108
min,0.000000,0.123165,0.005158,7.615773,7.141428,4.000000,0.000000,0.150029,0.002632,0.000000,0.067953,0.051291
25%,0.077646,0.544343,0.135176,26.688002,43.428101,9.380832,0.192795,0.767149,0.079380,0.090426,0.388172,0.500123
50%,0.390260,0.591799,0.461018,46.889221,52.191952,26.079691,0.492780,0.880685,0.358042,0.398277,0.432158,0.684654
75%,0.646230,0.663981,0.653210,69.995712,59.715996,57.984913,0.707122,0.936545,0.600438,0.786206,0.535261,0.742424
max,0.864660,0.780657,0.802046,106.323563,77.181602,99.676979,0.899105,0.989876,0.840174,0.938464,0.794648,1.000000


## With Supervision

In [ ]:
tta_sup_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"

tta_sup_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"

tta_sup_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [45]:
tta_sup_lr4 = pd.read_csv(tta_sup_lr4_paths)
tta_sup_lr4 = clean_df(tta_sup_lr4, True, 'brats')
tta_sup_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.347941,0.495431,0.365733,38.813123,43.713972,50.335952,0.326960,0.563019,0.308629,0.573271,0.482513,0.585556
std,0.334893,0.243776,0.262369,26.117581,19.044049,25.822028,0.329111,0.291597,0.254915,0.409836,0.231146,0.289584
min,0.000000,0.001936,0.000000,3.000000,7.000000,4.123106,0.000000,0.001276,0.000000,0.000000,0.003870,0.000000
25%,0.020761,0.326060,0.150536,17.130354,30.258068,33.721645,0.011371,0.336774,0.088381,0.028061,0.349850,0.472430
50%,0.242867,0.580779,0.350420,34.336571,46.754681,49.322891,0.161383,0.646370,0.250137,0.795838,0.506175,0.653338
75%,0.690403,0.675346,0.622812,55.015900,57.775429,66.228386,0.677950,0.792913,0.520246,0.923234,0.629470,0.807801
max,0.886455,0.845143,0.794053,105.488380,75.757515,104.073875,0.931949,0.976393,0.808489,1.000000,0.885142,0.971747


In [46]:
tta_sup_lr4_augs = pd.read_csv(tta_sup_lr4_augs_paths)
tta_sup_lr4_augs = clean_df(tta_sup_lr4_augs, True, 'brats')
tta_sup_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.247721,0.386213,0.485928,22.402572,26.309562,28.744791,0.170753,0.309408,0.441687,0.903240,0.662011,0.725310
std,0.249950,0.258291,0.277249,13.873788,13.149015,19.176580,0.195115,0.234943,0.283231,0.165786,0.290136,0.151698
min,0.002369,0.035094,0.003915,5.830952,10.344080,4.898980,0.001186,0.022150,0.001964,0.446885,0.084442,0.407700
25%,0.053563,0.189046,0.392666,9.611283,17.500000,12.638446,0.027777,0.116210,0.279684,0.905877,0.532233,0.672070
50%,0.147049,0.410681,0.550801,24.384280,22.594810,27.585952,0.080804,0.305551,0.500579,0.944803,0.781428,0.781719
75%,0.380065,0.607275,0.695860,28.866070,31.769564,42.566396,0.238596,0.459994,0.617385,0.994939,0.889258,0.802512
max,0.652803,0.696428,0.772107,42.877129,49.537865,56.349854,0.515925,0.661499,0.813225,1.000000,0.902850,0.896993


In [47]:
tta_sup_lr5 = pd.read_csv(tta_sup_lr5_paths)
tta_sup_lr5 = clean_df(tta_sup_lr5, True, 'brats')
tta_sup_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.372379,0.545874,0.378086,44.598101,48.129536,37.432518,0.406428,0.740669,0.315771,0.462604,0.448467,0.659987
std,0.302191,0.191885,0.277233,30.491796,19.264959,27.745940,0.311035,0.228090,0.266922,0.361521,0.178823,0.253797
min,0.000000,0.068077,0.000000,4.000000,7.141428,4.000000,0.000000,0.059917,0.000000,0.000000,0.069898,0.000000
25%,0.074834,0.416682,0.076260,15.012125,41.730450,12.077491,0.061033,0.631339,0.042256,0.056902,0.343596,0.530124
50%,0.383913,0.589241,0.421852,44.030874,52.182285,31.279312,0.397299,0.813463,0.296907,0.432374,0.450280,0.714286
75%,0.628618,0.696168,0.643630,65.036804,58.555200,55.018731,0.650804,0.913432,0.555347,0.834391,0.566861,0.840727
max,0.878523,0.814699,0.826087,106.228508,83.192551,98.639496,0.964245,0.989741,0.802792,0.974576,0.826952,1.000000


In [50]:
tta_sup_lr5_augs = pd.read_csv(tta_sup_lr5_augs_paths)
tta_sup_lr5_augs = clean_df(tta_sup_lr5_augs, True, 'brats')
tta_sup_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.262958,0.398295,0.363120,25.904093,33.066081,31.770966,0.218483,0.365820,0.299888,0.680438,0.631759,0.704135
std,0.217734,0.279432,0.284716,25.028929,22.416143,28.615752,0.243353,0.271624,0.264581,0.336665,0.305936,0.272552
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.061736,0.155459,0.063851,12.175678,19.437060,12.640184,0.032221,0.085833,0.033271,0.600867,0.543005,0.664428
50%,0.257644,0.397380,0.389825,15.757777,26.039936,20.868220,0.151277,0.433039,0.299953,0.817679,0.722514,0.766619
75%,0.370957,0.649507,0.628809,26.517834,43.071990,37.843587,0.264531,0.568289,0.550929,0.891990,0.851960,0.870742
max,0.630502,0.864862,0.750678,97.348854,97.164810,107.786827,0.819953,0.833447,0.717001,1.000000,0.915471,0.958042


In [51]:
tta_sup_lr6 = pd.read_csv(tta_sup_lr6_paths)
tta_sup_lr6 = clean_df(tta_sup_lr6, True, 'brats')
tta_sup_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.363392,0.527099,0.383779,48.468966,49.452761,36.671617,0.432953,0.777238,0.331066,0.416905,0.414660,0.639469
std,0.291539,0.177726,0.278586,29.313356,18.523090,26.885979,0.307123,0.210576,0.276832,0.348662,0.165667,0.250633
min,0.000000,0.120350,0.000097,5.830952,7.000000,4.000000,0.000000,0.174258,0.000051,0.000000,0.066232,0.000850
25%,0.059284,0.421046,0.081046,26.551121,42.289479,11.448597,0.093046,0.653491,0.044997,0.043956,0.300220,0.516145
50%,0.378088,0.574288,0.393701,47.277498,52.062014,34.332706,0.475136,0.853600,0.324970,0.347639,0.425856,0.700992
75%,0.601955,0.654594,0.639026,66.818272,60.046872,54.185463,0.697308,0.935897,0.558768,0.757368,0.511833,0.840021
max,0.871974,0.799162,0.838451,108.212212,81.418671,98.799797,0.965398,0.989312,0.831729,0.936945,0.782275,1.000000


In [54]:
tta_sup_lr6_augs = pd.read_csv(tta_sup_lr6_augs_paths)
tta_sup_lr6_augs = clean_df(tta_sup_lr6_augs, True, 'brats')
tta_sup_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.282276,0.411638,0.370360,21.991072,29.338166,27.373671,0.243738,0.411396,0.308080,0.694435,0.630206,0.730149
std,0.239290,0.271257,0.295229,20.370911,16.097224,23.498680,0.258833,0.292299,0.273716,0.293013,0.261030,0.262575
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043254,0.168976,0.034555,11.670138,19.195521,13.133801,0.022582,0.101512,0.017602,0.460642,0.499349,0.669745
50%,0.291933,0.450214,0.363765,13.638182,27.000000,18.000000,0.175494,0.456695,0.259329,0.764706,0.743800,0.777918
75%,0.487246,0.640401,0.651586,23.817457,34.923725,29.385889,0.394325,0.655143,0.568017,0.911967,0.824321,0.902062
max,0.675948,0.866894,0.782108,79.007599,73.155304,90.019440,0.871901,0.852970,0.786344,1.000000,0.884909,1.000000


In [174]:
tta_sup_first_layer_lr4_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_first_layer_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_first_layer_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_first_layer_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_first_layer_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_first_layer_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_first_layer_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [175]:
tta_sup_first_layer_lr4 = pd.read_csv(tta_sup_first_layer_lr4_paths)
tta_sup_first_layer_lr4 = clean_df(tta_sup_first_layer_lr4, True, 'brats')
tta_sup_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.305303,0.456571,0.395668,54.042513,52.127006,39.154772,0.375589,0.805793,0.337069,0.403137,0.337840,0.658735
std,0.286255,0.152963,0.253202,26.273229,15.784767,27.801023,0.321780,0.224423,0.251910,0.349275,0.139527,0.236317
min,0.000000,0.076111,0.000000,7.771520,6.403124,5.385165,0.000000,0.071188,0.000000,0.000000,0.058325,0.000000
25%,0.029767,0.390938,0.170192,33.421551,46.572525,15.264338,0.048611,0.763094,0.094986,0.049038,0.264410,0.566556
50%,0.246293,0.456072,0.443048,54.918121,54.460995,38.317081,0.302877,0.899496,0.363751,0.372087,0.313653,0.689876
75%,0.551663,0.565392,0.617933,70.763680,61.044231,57.671482,0.660984,0.949699,0.530212,0.724805,0.407812,0.805499
max,0.797493,0.794996,0.795810,105.312630,81.104874,113.021019,0.914928,0.992901,0.829081,1.000000,0.774589,1.000000


In [176]:
tta_sup_first_layer_lr4_augs = pd.read_csv(tta_sup_first_layer_lr4_augs_paths)
tta_sup_first_layer_lr4_augs = clean_df(tta_sup_first_layer_lr4_augs, True, 'brats')
tta_sup_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.255201,0.398441,0.401012,23.092861,29.618407,27.342576,0.178312,0.324257,0.339467,0.897110,0.692796,0.779981
std,0.267651,0.256148,0.254762,14.652744,17.236278,16.567608,0.203249,0.239935,0.262367,0.180222,0.231884,0.162715
min,0.001697,0.018102,0.024960,6.621739,6.082763,4.000000,0.000849,0.009212,0.012663,0.432669,0.188825,0.385892
25%,0.031464,0.198443,0.189843,9.999182,17.500000,14.421460,0.016207,0.142943,0.108999,0.927184,0.532071,0.693627
50%,0.114744,0.387459,0.389560,22.449944,25.019993,27.459061,0.061099,0.248111,0.258993,0.958188,0.810550,0.808129
75%,0.530687,0.632898,0.625454,29.993775,45.099037,37.696745,0.371717,0.528675,0.542422,0.992898,0.874600,0.905019
max,0.669985,0.771911,0.745727,57.442139,60.000000,56.320946,0.510268,0.709553,0.729350,1.000000,0.914464,0.986805


In [177]:
tta_sup_first_layer_lr5 = pd.read_csv(tta_sup_first_layer_lr5_paths)
tta_sup_first_layer_lr5 = clean_df(tta_sup_first_layer_lr5, True, 'brats')
tta_sup_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.372379,0.545874,0.378086,44.598101,48.129536,37.432518,0.406428,0.740669,0.315771,0.462604,0.448467,0.659987
std,0.302191,0.191885,0.277233,30.491796,19.264959,27.745940,0.311035,0.228090,0.266922,0.361521,0.178823,0.253797
min,0.000000,0.068077,0.000000,4.000000,7.141428,4.000000,0.000000,0.059917,0.000000,0.000000,0.069898,0.000000
25%,0.074834,0.416682,0.076260,15.012125,41.730450,12.077491,0.061033,0.631339,0.042256,0.056902,0.343596,0.530124
50%,0.383913,0.589241,0.421852,44.030874,52.182285,31.279312,0.397299,0.813463,0.296907,0.432374,0.450280,0.714286
75%,0.628618,0.696168,0.643630,65.036804,58.555200,55.018731,0.650804,0.913432,0.555347,0.834391,0.566861,0.840727
max,0.878523,0.814699,0.826087,106.228508,83.192551,98.639496,0.964245,0.989741,0.802792,0.974576,0.826952,1.000000


In [178]:
tta_sup_first_layer_lr5 = pd.read_csv(tta_sup_first_layer_lr5_paths)
tta_sup_first_layer_lr5 = clean_df(tta_sup_first_layer_lr5, True, 'brats')
tta_sup_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.372379,0.545874,0.378086,44.598101,48.129536,37.432518,0.406428,0.740669,0.315771,0.462604,0.448467,0.659987
std,0.302191,0.191885,0.277233,30.491796,19.264959,27.745940,0.311035,0.228090,0.266922,0.361521,0.178823,0.253797
min,0.000000,0.068077,0.000000,4.000000,7.141428,4.000000,0.000000,0.059917,0.000000,0.000000,0.069898,0.000000
25%,0.074834,0.416682,0.076260,15.012125,41.730450,12.077491,0.061033,0.631339,0.042256,0.056902,0.343596,0.530124
50%,0.383913,0.589241,0.421852,44.030874,52.182285,31.279312,0.397299,0.813463,0.296907,0.432374,0.450280,0.714286
75%,0.628618,0.696168,0.643630,65.036804,58.555200,55.018731,0.650804,0.913432,0.555347,0.834391,0.566861,0.840727
max,0.878523,0.814699,0.826087,106.228508,83.192551,98.639496,0.964245,0.989741,0.802792,0.974576,0.826952,1.000000


In [179]:
tta_sup_first_layer_lr5_augs = pd.read_csv(tta_sup_first_layer_lr5_augs_paths)
tta_sup_first_layer_lr5_augs = clean_df(tta_sup_first_layer_lr5_augs, True, 'brats')
tta_sup_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.269983,0.418161,0.371853,24.208024,33.007482,31.577187,0.226369,0.385186,0.308796,0.724641,0.611068,0.695734
std,0.224930,0.277960,0.287996,24.498859,23.717745,29.503854,0.249736,0.264704,0.268659,0.305146,0.304289,0.276011
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.055636,0.162584,0.088318,11.403426,18.645808,12.371552,0.028700,0.132046,0.046978,0.696948,0.515708,0.645718
50%,0.272942,0.498962,0.395990,14.599632,25.000000,21.000000,0.168799,0.479826,0.299913,0.841727,0.700517,0.767701
75%,0.410027,0.650104,0.652057,25.257650,45.491663,32.811720,0.329226,0.562950,0.557924,0.908091,0.839439,0.870788
max,0.625398,0.862905,0.751809,97.348854,97.421501,107.786827,0.824675,0.829799,0.740958,1.000000,0.911091,0.962601


In [180]:
tta_sup_first_layer_lr6 = pd.read_csv(tta_sup_first_layer_lr6_paths)
tta_sup_first_layer_lr6 = clean_df(tta_sup_first_layer_lr6, True, 'brats')
tta_sup_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.373467,0.546299,0.383304,44.353626,48.155324,37.533525,0.401501,0.743302,0.320891,0.470464,0.447924,0.663788
std,0.305238,0.191834,0.277834,30.329267,19.281931,27.782255,0.313096,0.229366,0.268796,0.363303,0.178104,0.251973
min,0.000000,0.073944,0.000000,4.000000,7.348469,4.000000,0.000000,0.065693,0.000000,0.000000,0.069176,0.000000
25%,0.066053,0.418992,0.087448,14.554399,41.459650,11.702639,0.061598,0.627073,0.047469,0.055587,0.346176,0.534307
50%,0.388473,0.593445,0.434830,44.338354,52.414423,31.252272,0.379682,0.818429,0.310555,0.463907,0.452276,0.706872
75%,0.634450,0.694610,0.644867,65.866276,58.774550,56.273520,0.656299,0.916245,0.562032,0.848357,0.561509,0.845195
max,0.881878,0.811574,0.826750,106.580719,83.628044,98.605270,0.964245,0.990487,0.808532,0.975207,0.824323,1.000000


In [181]:
tta_sup_first_layer_lr6_augs = pd.read_csv(tta_sup_first_layer_lr6_augs_paths)
tta_sup_first_layer_lr6_augs = clean_df(tta_sup_first_layer_lr6_augs, True, 'brats')
tta_sup_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.283721,0.412713,0.369906,21.949922,29.330738,27.474620,0.245168,0.412086,0.307226,0.696044,0.630463,0.730970
std,0.239978,0.270536,0.294830,20.354658,16.097257,23.583448,0.259379,0.291523,0.272957,0.289826,0.260931,0.262512
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043770,0.169190,0.035573,11.673383,19.157809,13.133801,0.022863,0.101064,0.018135,0.461422,0.499114,0.670186
50%,0.292994,0.450274,0.362980,13.638182,27.000000,18.000000,0.176368,0.456513,0.257912,0.767857,0.745252,0.779238
75%,0.487773,0.640259,0.651726,23.419199,34.918924,29.140294,0.396205,0.655486,0.566975,0.911869,0.823901,0.901004
max,0.679971,0.866661,0.781698,79.036385,73.177864,90.019440,0.872491,0.852379,0.784173,1.000000,0.884671,1.000000


In [63]:
tta_sup_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_norm_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_norm_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_norm_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_norm_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [64]:
tta_sup_norm_lr4 = pd.read_csv(tta_sup_norm_lr4_paths)
tta_sup_norm_lr4 = clean_df(tta_sup_norm_lr4, True, 'brats')
tta_sup_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.343624,0.509897,0.373086,37.743865,42.797396,50.401930,0.328664,0.580005,0.310435,0.572216,0.495170,0.587461
std,0.341289,0.238515,0.261292,25.601984,19.315719,26.769226,0.336266,0.287005,0.245847,0.417321,0.226276,0.291360
min,0.000000,0.002906,0.000000,3.000000,6.164414,4.000000,0.000000,0.001626,0.000000,0.000000,0.009732,0.000000
25%,0.003554,0.317397,0.167205,18.357559,30.753841,34.000732,0.001988,0.366067,0.093221,0.027814,0.354499,0.428394
50%,0.185812,0.585713,0.353792,34.098297,45.287968,49.887371,0.161383,0.659719,0.252734,0.791367,0.526637,0.669555
75%,0.718404,0.694007,0.623388,50.600388,56.542019,65.760910,0.680247,0.814297,0.534355,0.940555,0.629376,0.806448
max,0.894669,0.848025,0.822017,104.639130,76.347885,105.529121,0.925029,0.976448,0.737554,1.000000,0.885330,1.000000


In [65]:
tta_sup_norm_lr4_augs = pd.read_csv(tta_sup_norm_lr4_augs_paths)
tta_sup_norm_lr4_augs = clean_df(tta_sup_norm_lr4_augs, True, 'brats')
tta_sup_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,0.231817,0.361879,0.411299,25.136999,30.989784,26.346978,0.158442,0.291132,0.363405,0.917803,0.664557,0.752668
std,0.241532,0.251637,0.285636,16.214503,17.123211,17.999848,0.185731,0.234326,0.287367,0.144422,0.263900,0.180692
min,0.000485,0.051620,0.019576,6.164414,9.433981,4.898980,0.000243,0.027259,0.009993,0.481403,0.135719,0.405796
25%,0.052732,0.151115,0.114621,9.879971,18.750000,9.500000,0.027191,0.083417,0.061163,0.917706,0.523049,0.683427
50%,0.172879,0.270330,0.510592,26.019253,25.000000,27.069229,0.095564,0.215273,0.423146,0.962817,0.803431,0.754738
75%,0.317169,0.625700,0.667238,31.238254,48.510078,34.985531,0.206758,0.487563,0.583503,0.996725,0.864041,0.910473
max,0.629512,0.694571,0.744454,58.027573,60.008331,56.329388,0.468737,0.665875,0.784400,1.000000,0.895665,0.982964


In [66]:
tta_sup_norm_lr5 = pd.read_csv(tta_sup_norm_lr5_paths)
tta_sup_norm_lr5 = clean_df(tta_sup_norm_lr5, True, 'brats')
tta_sup_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.371248,0.546171,0.382568,44.508675,48.024380,37.514935,0.401879,0.742694,0.320770,0.464220,0.448202,0.661299
std,0.303547,0.191880,0.277902,30.366709,19.460676,27.776626,0.311095,0.228357,0.269068,0.363856,0.178791,0.251905
min,0.000000,0.072060,0.000000,4.000000,7.280110,4.000000,0.000000,0.063506,0.000000,0.000000,0.069069,0.000000
25%,0.071775,0.419850,0.084767,14.735693,41.477815,11.660972,0.060991,0.626592,0.046300,0.054534,0.344443,0.548902
50%,0.371098,0.593294,0.432044,44.363314,52.540226,31.969441,0.386428,0.816845,0.308929,0.446406,0.450528,0.705787
75%,0.627380,0.696605,0.644497,65.809490,58.670546,55.901784,0.657172,0.915303,0.560464,0.844658,0.564619,0.842405
max,0.880814,0.815839,0.826018,106.485886,83.060219,98.631134,0.964245,0.989638,0.810840,0.975000,0.826845,1.000000


In [67]:
tta_sup_norm_lr5_augs = pd.read_csv(tta_sup_norm_lr5_augs_paths)
tta_sup_norm_lr5_augs = clean_df(tta_sup_norm_lr5_augs, True, 'brats')
tta_sup_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.242358,0.389786,0.369075,24.336210,32.782463,30.598903,0.202315,0.357976,0.305766,0.706189,0.614464,0.706531
std,0.222765,0.280039,0.292960,23.212343,22.037707,28.458961,0.242384,0.269909,0.267913,0.321142,0.294503,0.265288
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044233,0.158046,0.060623,12.328828,18.000000,12.000000,0.022618,0.086549,0.031584,0.668920,0.478917,0.664002
50%,0.253826,0.305580,0.400768,16.430161,27.013885,20.904545,0.147950,0.393261,0.302242,0.830619,0.704309,0.760396
75%,0.326550,0.652630,0.690230,25.079872,45.000000,29.427877,0.213844,0.565867,0.561912,0.919106,0.834365,0.866893
max,0.628134,0.862618,0.741728,97.348854,97.421501,107.786827,0.809327,0.830043,0.718616,1.000000,0.908754,0.962754


In [68]:
tta_sup_norm_lr6 = pd.read_csv(tta_sup_norm_lr6_paths)
tta_sup_norm_lr6 = clean_df(tta_sup_norm_lr6, True, 'brats')
tta_sup_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.363866,0.527320,0.383741,48.432863,49.448911,36.670267,0.433028,0.777186,0.330878,0.417418,0.414948,0.640254
std,0.291818,0.177777,0.278635,29.325089,18.530196,26.900303,0.306992,0.210638,0.276832,0.349048,0.165803,0.250175
min,0.000000,0.120290,0.000097,5.830952,7.000000,4.000000,0.000000,0.173913,0.000051,0.000000,0.066191,0.000862
25%,0.059197,0.421284,0.080911,26.552540,42.295314,11.436894,0.093526,0.654134,0.044932,0.043787,0.300455,0.519207
50%,0.378640,0.574524,0.393772,47.017838,52.053905,34.332706,0.475925,0.853353,0.324345,0.349681,0.425839,0.701451
75%,0.604356,0.654241,0.638651,66.747999,60.022432,54.195617,0.698132,0.935825,0.559672,0.759547,0.511897,0.842313
max,0.872344,0.799210,0.838523,108.226761,81.443230,98.808144,0.965398,0.989233,0.831820,0.936807,0.783879,1.000000


In [69]:
tta_sup_norm_lr6_augs = pd.read_csv(tta_sup_norm_lr6_augs_paths)
tta_sup_norm_lr6_augs = clean_df(tta_sup_norm_lr6_augs, True, 'brats')
tta_sup_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.283295,0.411803,0.370331,21.975305,29.326564,27.495053,0.244538,0.411508,0.307929,0.695858,0.630579,0.730322
std,0.239254,0.271201,0.295150,20.361024,16.089811,23.584666,0.258706,0.292311,0.273440,0.289674,0.261033,0.262580
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043312,0.169298,0.034490,11.651594,19.182966,13.114821,0.022617,0.101757,0.017602,0.460487,0.499248,0.669454
50%,0.292167,0.450903,0.363981,13.638182,27.000000,18.000000,0.175809,0.456518,0.259532,0.764706,0.744941,0.778423
75%,0.487317,0.640403,0.651760,23.824287,34.918924,29.391495,0.393323,0.655608,0.568620,0.912695,0.824209,0.902313
max,0.678103,0.866631,0.782142,79.027206,73.155304,90.019440,0.870720,0.852277,0.785897,1.000000,0.885124,1.000000


In [70]:
tta_sup_decoder_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_decoder_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_decoder_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_decoder_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_decoder_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_sup_decoder_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [71]:
tta_sup_decoder_lr6_augs = pd.read_csv(tta_sup_decoder_lr6_augs_paths)
tta_sup_decoder_lr6_augs = clean_df(tta_sup_decoder_lr6_augs, True, 'brats')
tta_sup_decoder_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.282276,0.411638,0.370360,21.991072,29.338166,27.373671,0.243738,0.411396,0.308080,0.694435,0.630206,0.730149
std,0.239290,0.271257,0.295229,20.370911,16.097224,23.498680,0.258833,0.292299,0.273716,0.293013,0.261030,0.262575
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043254,0.168976,0.034555,11.670138,19.195521,13.133801,0.022582,0.101512,0.017602,0.460642,0.499349,0.669745
50%,0.291933,0.450214,0.363765,13.638182,27.000000,18.000000,0.175494,0.456695,0.259329,0.764706,0.743800,0.777918
75%,0.487246,0.640401,0.651586,23.817457,34.923725,29.385889,0.394325,0.655143,0.568017,0.911967,0.824321,0.902062
max,0.675948,0.866894,0.782108,79.007599,73.155304,90.019440,0.871901,0.852970,0.786344,1.000000,0.884909,1.000000


In [72]:
tta_sup_decoder_lr6 = pd.read_csv(tta_sup_decoder_lr6_paths)
tta_sup_decoder_lr6 = clean_df(tta_sup_decoder_lr6, True, 'brats')
tta_sup_decoder_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.363392,0.527099,0.383779,48.468966,49.452761,36.671617,0.432953,0.777238,0.331066,0.416905,0.414660,0.639469
std,0.291539,0.177726,0.278586,29.313356,18.523090,26.885979,0.307123,0.210576,0.276832,0.348662,0.165667,0.250633
min,0.000000,0.120350,0.000097,5.830952,7.000000,4.000000,0.000000,0.174258,0.000051,0.000000,0.066232,0.000850
25%,0.059284,0.421046,0.081046,26.551121,42.289479,11.448597,0.093046,0.653491,0.044997,0.043956,0.300220,0.516145
50%,0.378088,0.574288,0.393701,47.277498,52.062014,34.332706,0.475136,0.853600,0.324970,0.347639,0.425856,0.700992
75%,0.601955,0.654594,0.639026,66.818272,60.046872,54.185463,0.697308,0.935897,0.558768,0.757368,0.511833,0.840021
max,0.871974,0.799162,0.838451,108.212212,81.418671,98.799797,0.965398,0.989312,0.831729,0.936945,0.782275,1.000000


In [73]:
tta_sup_decoder_lr5_augs = pd.read_csv(tta_sup_decoder_lr5_augs_paths)
tta_sup_decoder_lr5_augs = clean_df(tta_sup_decoder_lr5_augs, True, 'brats')
tta_sup_decoder_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.262958,0.398295,0.363120,25.904093,33.066081,31.770966,0.218483,0.365820,0.299888,0.680438,0.631759,0.704135
std,0.217734,0.279432,0.284716,25.028929,22.416143,28.615752,0.243353,0.271624,0.264581,0.336665,0.305936,0.272552
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.061736,0.155459,0.063851,12.175678,19.437060,12.640184,0.032221,0.085833,0.033271,0.600867,0.543005,0.664428
50%,0.257644,0.397380,0.389825,15.757777,26.039936,20.868220,0.151277,0.433039,0.299953,0.817679,0.722514,0.766619
75%,0.370957,0.649507,0.628809,26.517834,43.071990,37.843587,0.264531,0.568289,0.550929,0.891990,0.851960,0.870742
max,0.630502,0.864862,0.750678,97.348854,97.164810,107.786827,0.819953,0.833447,0.717001,1.000000,0.915471,0.958042


In [74]:
tta_sup_decoder_lr5 = pd.read_csv(tta_sup_decoder_lr5_paths)
tta_sup_decoder_lr5 = clean_df(tta_sup_decoder_lr5, True, 'brats')
tta_sup_decoder_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.372379,0.545874,0.378086,44.598101,48.129536,37.432518,0.406428,0.740669,0.315771,0.462604,0.448467,0.659987
std,0.302191,0.191885,0.277233,30.491796,19.264959,27.745940,0.311035,0.228090,0.266922,0.361521,0.178823,0.253797
min,0.000000,0.068077,0.000000,4.000000,7.141428,4.000000,0.000000,0.059917,0.000000,0.000000,0.069898,0.000000
25%,0.074834,0.416682,0.076260,15.012125,41.730450,12.077491,0.061033,0.631339,0.042256,0.056902,0.343596,0.530124
50%,0.383913,0.589241,0.421852,44.030874,52.182285,31.279312,0.397299,0.813463,0.296907,0.432374,0.450280,0.714286
75%,0.628618,0.696168,0.643630,65.036804,58.555200,55.018731,0.650804,0.913432,0.555347,0.834391,0.566861,0.840727
max,0.878523,0.814699,0.826087,106.228508,83.192551,98.639496,0.964245,0.989741,0.802792,0.974576,0.826952,1.000000


In [75]:
tta_sup_decoder_lr4 = pd.read_csv(tta_sup_decoder_lr4_paths)
tta_sup_decoder_lr4 = clean_df(tta_sup_decoder_lr4, True, 'brats')
tta_sup_decoder_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.347941,0.495431,0.365733,38.813123,43.713972,50.335952,0.326960,0.563019,0.308629,0.573271,0.482513,0.585556
std,0.334893,0.243776,0.262369,26.117581,19.044049,25.822028,0.329111,0.291597,0.254915,0.409836,0.231146,0.289584
min,0.000000,0.001936,0.000000,3.000000,7.000000,4.123106,0.000000,0.001276,0.000000,0.000000,0.003870,0.000000
25%,0.020761,0.326060,0.150536,17.130354,30.258068,33.721645,0.011371,0.336774,0.088381,0.028061,0.349850,0.472430
50%,0.242867,0.580779,0.350420,34.336571,46.754681,49.322891,0.161383,0.646370,0.250137,0.795838,0.506175,0.653338
75%,0.690403,0.675346,0.622812,55.015900,57.775429,66.228386,0.677950,0.792913,0.520246,0.923234,0.629470,0.807801
max,0.886455,0.845143,0.794053,105.488380,75.757515,104.073875,0.931949,0.976393,0.808489,1.000000,0.885142,0.971747


In [76]:
tta_sup_decoder_lr4_augs = pd.read_csv(tta_sup_decoder_lr4_augs_paths)
tta_sup_decoder_lr4_augs = clean_df(tta_sup_decoder_lr4_augs, True, 'brats')
tta_sup_decoder_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.247721,0.386213,0.485928,22.402572,26.309562,28.744791,0.170753,0.309408,0.441687,0.903240,0.662011,0.725310
std,0.249950,0.258291,0.277249,13.873788,13.149015,19.176580,0.195115,0.234943,0.283231,0.165786,0.290136,0.151698
min,0.002369,0.035094,0.003915,5.830952,10.344080,4.898980,0.001186,0.022150,0.001964,0.446885,0.084442,0.407700
25%,0.053563,0.189046,0.392666,9.611283,17.500000,12.638446,0.027777,0.116210,0.279684,0.905877,0.532233,0.672070
50%,0.147049,0.410681,0.550801,24.384280,22.594810,27.585952,0.080804,0.305551,0.500579,0.944803,0.781428,0.781719
75%,0.380065,0.607275,0.695860,28.866070,31.769564,42.566396,0.238596,0.459994,0.617385,0.994939,0.889258,0.802512
max,0.652803,0.696428,0.772107,42.877129,49.537865,56.349854,0.515925,0.661499,0.813225,1.000000,0.902850,0.896993


### LR 4

In [77]:
mask = tta_sup_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.453884,0.554697,0.373311,38.188111,45.426733,49.612533,0.407333,0.644818,0.319591,0.633745,0.528852,0.628594
std,0.346997,0.214227,0.253640,25.939603,20.245010,27.015216,0.333175,0.262669,0.261215,0.386634,0.208397,0.245131
min,0.000000,0.006660,0.000000,3.000000,7.000000,4.123106,0.000000,0.006091,0.000000,0.000000,0.007346,0.000000
25%,0.102052,0.512958,0.159329,23.600847,35.566837,24.041630,0.064775,0.539795,0.088381,0.244416,0.481233,0.553949
50%,0.512977,0.627026,0.350420,34.336571,47.968739,51.054848,0.403349,0.742265,0.256265,0.805397,0.573655,0.674782
75%,0.816748,0.690289,0.622812,54.018494,60.876926,66.228386,0.744879,0.816923,0.520246,0.923234,0.629470,0.790572
max,0.886455,0.804104,0.753643,105.488380,75.757515,104.073875,0.868412,0.976393,0.808489,0.987811,0.885142,0.971747


In [78]:
mask = tta_sup_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.398813,0.529935,0.418041,34.887367,40.214312,26.207798,0.451349,0.844692,0.363435,0.524569,0.409572,0.674249
std,0.327574,0.167569,0.264667,26.787275,21.951498,27.926907,0.341906,0.173149,0.270350,0.418283,0.176611,0.238871
min,0.000000,0.119377,0.000000,4.000000,7.549834,5.000000,0.000000,0.187761,0.000000,0.000000,0.065714,0.000000
25%,0.019171,0.434392,0.247709,9.899495,19.723083,7.280110,0.009678,0.804496,0.161547,0.046972,0.286842,0.562364
50%,0.403662,0.494986,0.396193,29.017237,41.424629,13.453624,0.495812,0.915466,0.363625,0.638380,0.359304,0.725324
75%,0.752336,0.657340,0.651400,52.354561,59.539902,39.440445,0.764076,0.955399,0.591151,0.968087,0.502047,0.787957
max,0.856765,0.816531,0.852126,103.670151,78.192070,113.021019,0.914928,0.992901,0.828995,1.000000,0.817408,0.990136


In [79]:
mask = tta_sup_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.449747,0.567729,0.375568,35.269813,45.243996,49.469898,0.411497,0.663092,0.310584,0.623924,0.535711,0.627706
std,0.356690,0.204242,0.247388,25.561231,20.268103,28.425362,0.342714,0.251375,0.238355,0.397540,0.201561,0.258840
min,0.000000,0.008933,0.000000,3.000000,6.164414,4.000000,0.000000,0.008255,0.000000,0.000000,0.009732,0.000000
25%,0.041029,0.551297,0.206193,11.789826,35.680496,23.413664,0.062893,0.595720,0.116796,0.188343,0.490491,0.531391
50%,0.536465,0.637743,0.353792,32.956779,48.187134,49.960960,0.486447,0.735669,0.252734,0.837762,0.559135,0.693069
75%,0.812191,0.697688,0.623388,43.216778,60.216278,63.725975,0.740062,0.836834,0.534355,0.939954,0.629376,0.801989
max,0.894669,0.817591,0.738692,104.639130,76.347885,105.529121,0.920033,0.976448,0.717165,0.992501,0.885330,1.000000


### L5

In [80]:
mask = tta_sup_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.433132,0.583899,0.397663,47.144047,47.826167,35.852410,0.451943,0.786629,0.342729,0.485221,0.482865,0.655037
std,0.325013,0.165285,0.266152,29.116199,19.908918,29.426153,0.314059,0.196577,0.269495,0.368874,0.164106,0.224200
min,0.000000,0.118938,0.001433,4.000000,7.141428,4.000000,0.000000,0.127581,0.000725,0.000000,0.069898,0.048874
25%,0.082404,0.554020,0.165469,26.765646,43.646305,8.654428,0.206093,0.725958,0.100767,0.167684,0.412035,0.558144
50%,0.439066,0.620914,0.434519,44.812382,53.385391,28.149603,0.513428,0.836911,0.355728,0.517341,0.474930,0.700385
75%,0.704736,0.698531,0.639571,64.451530,60.934391,54.967258,0.716981,0.913683,0.575751,0.863744,0.566941,0.808647
max,0.878523,0.775742,0.826087,103.637825,77.446754,98.639496,0.893815,0.989741,0.802792,0.958940,0.826952,1.000000


In [81]:
mask = tta_sup_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.424580,0.625956,0.389427,35.201649,29.743414,28.969003,0.428241,0.783512,0.332097,0.510060,0.540128,0.690869
std,0.342426,0.180822,0.269926,34.066437,23.800243,27.904109,0.329876,0.201294,0.271681,0.413372,0.180731,0.241712
min,0.000000,0.128133,0.000000,4.000000,4.898980,4.000000,0.000000,0.123367,0.000000,0.000000,0.087113,0.000000
25%,0.065353,0.567250,0.156828,7.071068,8.602325,9.505208,0.071901,0.725753,0.092129,0.037843,0.447153,0.626593
50%,0.472473,0.686586,0.414731,21.563858,17.920654,15.066519,0.506300,0.835785,0.326420,0.637264,0.579437,0.766787
75%,0.711226,0.752009,0.642704,52.125328,50.447994,43.307606,0.716981,0.913558,0.559737,0.911824,0.647693,0.829106
max,0.889603,0.825966,0.836687,103.639519,78.232338,89.117615,0.893389,0.989741,0.802792,0.995932,0.834071,1.000000


In [82]:
mask = tta_sup_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.432448,0.584654,0.401397,47.156749,47.736761,35.779256,0.448912,0.789056,0.346401,0.485390,0.483155,0.656533
std,0.326878,0.165859,0.264932,29.246400,19.982577,29.433152,0.315799,0.196625,0.269985,0.370350,0.164836,0.224522
min,0.000000,0.114427,0.000828,4.000000,7.280110,4.000000,0.000000,0.122161,0.000420,0.000000,0.069069,0.031250
25%,0.073433,0.554879,0.173904,26.851442,42.000000,8.826048,0.180519,0.727834,0.107289,0.148012,0.413562,0.556282
50%,0.437724,0.619434,0.441280,45.099888,53.235325,27.092434,0.511605,0.840501,0.361183,0.517864,0.480859,0.696140
75%,0.714777,0.699355,0.642504,65.115280,60.975407,55.253960,0.725557,0.915337,0.570645,0.869288,0.565892,0.806371
max,0.880814,0.776213,0.825047,103.727470,77.485481,98.631134,0.892650,0.989638,0.810840,0.960216,0.826845,1.000000


In [83]:
mask = tta_sup_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.276798,0.416903,0.381224,24.211660,32.753770,31.179964,0.229983,0.383867,0.315163,0.716251,0.616827,0.690780
std,0.214472,0.274062,0.280441,24.510829,22.985649,29.274218,0.244375,0.266463,0.262615,0.304244,0.306740,0.273215
min,0.000000,0.000000,0.000000,5.099020,5.099020,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.099888,0.163816,0.134345,12.144801,18.874120,12.241964,0.053075,0.108717,0.074279,0.661206,0.520822,0.656585
50%,0.257971,0.507344,0.406822,14.456832,25.079872,20.736441,0.151618,0.468188,0.308474,0.842675,0.710969,0.759344
75%,0.412869,0.652016,0.650511,25.051970,43.847406,32.776330,0.311679,0.572397,0.558468,0.900920,0.836906,0.856807
max,0.630502,0.864862,0.750678,97.348854,97.164810,107.786827,0.819953,0.833447,0.717001,1.000000,0.911093,0.958042


In [84]:
mask = tta_sup_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.269983,0.418161,0.371853,24.208024,33.007482,31.577187,0.226369,0.385186,0.308796,0.724641,0.611068,0.695734
std,0.224930,0.277960,0.287996,24.498859,23.717745,29.503854,0.249736,0.264704,0.268659,0.305146,0.304289,0.276011
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.055636,0.162584,0.088318,11.403426,18.645808,12.371552,0.028700,0.132046,0.046978,0.696948,0.515708,0.645718
50%,0.272942,0.498962,0.395990,14.599632,25.000000,21.000000,0.168799,0.479826,0.299913,0.841727,0.700517,0.767701
75%,0.410027,0.650104,0.652057,25.257650,45.491663,32.811720,0.329226,0.562950,0.557924,0.908091,0.839439,0.870788
max,0.625398,0.862905,0.751809,97.348854,97.421501,107.786827,0.824675,0.829799,0.740958,1.000000,0.911091,0.962601


In [85]:
mask = tta_sup_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.267209,0.412919,0.368627,24.375203,32.806953,31.713637,0.223274,0.382385,0.304942,0.719726,0.612166,0.695255
std,0.219755,0.281359,0.286661,24.464795,23.229616,29.647988,0.245782,0.270248,0.264233,0.304523,0.303235,0.276538
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.060073,0.164081,0.067852,11.952332,17.746428,12.287919,0.031230,0.108954,0.035417,0.679684,0.520911,0.648047
50%,0.257143,0.504710,0.400768,14.866069,25.039968,20.904545,0.153322,0.469148,0.302242,0.830619,0.704309,0.752060
75%,0.412865,0.654062,0.649498,25.172709,45.319948,35.189223,0.315004,0.567581,0.551189,0.905836,0.836655,0.870935
max,0.628134,0.862618,0.741728,97.348854,97.421501,107.786827,0.809327,0.830043,0.718616,1.000000,0.908754,0.962754


### L6

In [86]:
mask = tta_sup_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.409755,0.564114,0.402673,50.444594,48.828388,36.286016,0.456251,0.811985,0.362493,0.448584,0.449625,0.631074
std,0.315898,0.156017,0.267906,28.339784,19.330045,28.551493,0.313233,0.183497,0.281058,0.365260,0.155203,0.212002
min,0.000000,0.120350,0.006239,7.000000,7.141428,4.000000,0.000000,0.174258,0.003166,0.000000,0.066232,0.063205
25%,0.078604,0.550505,0.119785,27.367865,42.965103,8.185352,0.215835,0.734516,0.068094,0.071366,0.390081,0.510733
50%,0.397506,0.598100,0.469653,47.391983,52.278103,28.248894,0.486441,0.888582,0.350137,0.421422,0.434552,0.682316
75%,0.659081,0.660204,0.645008,65.764732,60.315834,54.414146,0.707120,0.936957,0.593435,0.841438,0.541353,0.750415
max,0.871974,0.758825,0.804974,108.212212,77.549980,98.799797,0.902594,0.989312,0.831729,0.936945,0.782275,1.000000


In [87]:
mask = tta_sup_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.436312,0.584971,0.402132,46.912288,47.747485,35.842068,0.450625,0.791061,0.346183,0.490247,0.482637,0.659100
std,0.328480,0.165755,0.265172,29.191555,19.957681,29.421734,0.318088,0.196915,0.269801,0.369700,0.164112,0.225115
min,0.000000,0.115730,0.000753,4.000000,7.348469,4.000000,0.000000,0.123501,0.000381,0.000000,0.069176,0.028369
25%,0.072192,0.554990,0.177769,26.733871,42.000000,9.486833,0.162467,0.737542,0.110043,0.131643,0.415587,0.556279
50%,0.436209,0.619892,0.440919,45.693546,53.488316,27.202942,0.518070,0.840946,0.352231,0.539141,0.476944,0.696577
75%,0.727389,0.697348,0.643840,64.815125,60.967205,55.470715,0.731847,0.916303,0.587676,0.869022,0.562941,0.806958
max,0.881878,0.778521,0.825675,103.637825,77.485481,98.605270,0.895120,0.990487,0.808532,0.959964,0.824323,1.000000


In [88]:
mask = tta_sup_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.410116,0.564386,0.402671,50.401135,48.816289,36.284354,0.456316,0.811992,0.362217,0.448769,0.449990,0.631880
std,0.316201,0.156102,0.267807,28.362429,19.340338,28.557563,0.313263,0.183520,0.280936,0.365448,0.155425,0.211977
min,0.000000,0.120290,0.006127,6.708204,7.141428,4.000000,0.000000,0.173913,0.003109,0.000000,0.066191,0.063850
25%,0.078188,0.550835,0.119711,27.298344,43.000000,7.870832,0.215499,0.734210,0.068028,0.071094,0.390828,0.512770
50%,0.398130,0.598328,0.469098,46.865768,52.252243,28.248894,0.485788,0.888050,0.349217,0.420738,0.435357,0.682365
75%,0.658466,0.659655,0.644531,65.795135,60.291382,54.415070,0.703883,0.936819,0.592998,0.842570,0.540667,0.750277
max,0.872344,0.758972,0.804744,108.226761,77.548996,98.808144,0.902559,0.989233,0.831820,0.936807,0.783879,1.000000


In [89]:
mask = tta_sup_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.272326,0.427358,0.348435,23.361283,30.056689,30.119083,0.241078,0.420465,0.284327,0.686335,0.643000,0.725464
std,0.214242,0.281598,0.288467,21.990661,17.146526,24.750334,0.259724,0.295937,0.260462,0.274371,0.263692,0.286866
min,0.000000,0.000000,0.000000,5.744563,5.099020,4.898980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.075789,0.168976,0.034555,11.884559,19.195521,14.663635,0.039991,0.101512,0.017602,0.460642,0.544760,0.669745
50%,0.291933,0.547310,0.311251,13.638182,27.000000,20.314991,0.175494,0.515539,0.211546,0.755881,0.743800,0.779708
75%,0.462059,0.661077,0.608005,25.048855,41.005434,35.131953,0.352586,0.655143,0.532614,0.882629,0.825154,0.946977
max,0.627893,0.866894,0.739580,79.007599,73.155304,90.019440,0.871901,0.852970,0.664484,1.000000,0.884909,1.000000


In [90]:
mask = tta_sup_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.273642,0.428545,0.347976,23.311470,30.048203,30.247763,0.242386,0.421273,0.283546,0.686443,0.643125,0.726013
std,0.214936,0.280740,0.287994,21.975528,17.147079,24.827159,0.260095,0.295073,0.259697,0.274301,0.263803,0.286834
min,0.000000,0.000000,0.000000,5.744563,5.099020,4.898980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.074035,0.169190,0.035573,11.887804,19.157809,14.652038,0.039023,0.101064,0.018135,0.461422,0.544362,0.670186
50%,0.292994,0.547282,0.311164,13.638182,27.000000,20.322401,0.176368,0.515539,0.211309,0.762715,0.745252,0.780227
75%,0.461883,0.661349,0.607806,24.811012,41.005434,36.351332,0.353485,0.655486,0.530943,0.882802,0.825005,0.945561
max,0.629381,0.866661,0.737809,79.036385,73.177864,90.019440,0.872491,0.852379,0.664643,1.000000,0.884671,1.000000


In [91]:
mask = tta_sup_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.273252,0.427469,0.348407,23.342197,30.043151,30.268017,0.241806,0.420516,0.284196,0.686022,0.643308,0.725626
std,0.214167,0.281548,0.288390,21.980722,17.138759,24.831422,0.259477,0.295895,0.260192,0.274248,0.263810,0.286864
min,0.000000,0.000000,0.000000,5.744563,5.099020,4.898980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.076065,0.169298,0.034369,11.866015,19.182966,14.645329,0.040148,0.101757,0.017506,0.460487,0.544626,0.669454
50%,0.292167,0.547267,0.311921,13.638182,27.000000,20.314991,0.175809,0.515687,0.211996,0.755937,0.744941,0.779522
75%,0.461998,0.661374,0.608266,25.051970,41.005434,36.596927,0.351861,0.655608,0.532563,0.883042,0.825012,0.947190
max,0.628187,0.866631,0.737896,79.027206,73.155304,90.019440,0.870720,0.852277,0.663892,1.000000,0.885124,1.000000


## With Consistency

In [182]:
tta_con_lr4_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_lr4_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_lr5_paths =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_lr5_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_lr6_paths =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_lr6_augs_paths  =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [183]:
mask = tta_sup_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.410116,0.564386,0.402671,50.401135,48.816289,36.284354,0.456316,0.811992,0.362217,0.448769,0.449990,0.631880
std,0.316201,0.156102,0.267807,28.362429,19.340338,28.557563,0.313263,0.183520,0.280936,0.365448,0.155425,0.211977
min,0.000000,0.120290,0.006127,6.708204,7.141428,4.000000,0.000000,0.173913,0.003109,0.000000,0.066191,0.063850
25%,0.078188,0.550835,0.119711,27.298344,43.000000,7.870832,0.215499,0.734210,0.068028,0.071094,0.390828,0.512770
50%,0.398130,0.598328,0.469098,46.865768,52.252243,28.248894,0.485788,0.888050,0.349217,0.420738,0.435357,0.682365
75%,0.658466,0.659655,0.644531,65.795135,60.291382,54.415070,0.703883,0.936819,0.592998,0.842570,0.540667,0.750277
max,0.872344,0.758972,0.804744,108.226761,77.548996,98.808144,0.902559,0.989233,0.831820,0.936807,0.783879,1.000000


In [184]:
tta_con_lr4 = pd.read_csv(tta_con_lr4_paths)
tta_con_lr4 = clean_df(tta_con_lr4, True, 'brats')
tta_con_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.305303,0.456571,0.395668,54.042513,52.127006,39.154772,0.375589,0.805793,0.337069,0.403137,0.337840,0.658735
std,0.286255,0.152963,0.253202,26.273229,15.784767,27.801023,0.321780,0.224423,0.251910,0.349275,0.139527,0.236317
min,0.000000,0.076111,0.000000,7.771520,6.403124,5.385165,0.000000,0.071188,0.000000,0.000000,0.058325,0.000000
25%,0.029767,0.390938,0.170192,33.421551,46.572525,15.264338,0.048611,0.763094,0.094986,0.049038,0.264410,0.566556
50%,0.246293,0.456072,0.443048,54.918121,54.460995,38.317081,0.302877,0.899496,0.363751,0.372087,0.313653,0.689876
75%,0.551663,0.565392,0.617933,70.763680,61.044231,57.671482,0.660984,0.949699,0.530212,0.724805,0.407812,0.805499
max,0.797493,0.794996,0.795810,105.312630,81.104874,113.021019,0.914928,0.992901,0.829081,1.000000,0.774589,1.000000


In [185]:
tta_con_lr4_augs = pd.read_csv(tta_con_lr4_augs_paths)
tta_con_lr4_augs = clean_df(tta_con_lr4_augs, True, 'brats')
tta_con_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.269599,0.469746,0.485806,23.816606,38.063743,24.900747,0.225113,0.510817,0.442177,0.496815,0.547975,0.714653
std,0.283936,0.231000,0.274149,21.438738,19.748247,24.991646,0.272368,0.307644,0.291143,0.351086,0.236813,0.238301
min,0.000000,0.000000,0.000000,5.196152,11.357817,3.162278,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.020206,0.365492,0.208226,12.031196,23.801314,8.868195,0.010651,0.295904,0.118153,0.233950,0.441208,0.637021
50%,0.100154,0.545573,0.564087,15.504597,34.660604,14.764784,0.056009,0.495356,0.514014,0.458938,0.545924,0.752919
75%,0.501403,0.643569,0.724404,28.721920,46.823400,29.633538,0.394981,0.788930,0.720947,0.851126,0.706156,0.885157
max,0.789069,0.742000,0.847073,95.905159,94.005318,107.916634,0.914116,0.947865,0.865657,0.985269,0.945971,0.991698


In [186]:
tta_con_lr5 = pd.read_csv(tta_con_lr5_paths)
tta_con_lr5 = clean_df(tta_con_lr5, True, 'brats')
tta_con_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.376115,0.556507,0.392487,44.978425,47.683831,38.736627,0.399058,0.733351,0.331468,0.482186,0.467221,0.675070
std,0.305364,0.193428,0.277052,28.108807,18.097234,27.904927,0.316816,0.229116,0.270387,0.361946,0.184991,0.243281
min,0.000000,0.074081,0.000000,5.000000,7.280110,4.472136,0.000000,0.061329,0.000000,0.000000,0.066741,0.000000
25%,0.075065,0.436652,0.102163,26.000000,42.166336,11.575837,0.049338,0.614438,0.058241,0.066858,0.358320,0.548757
50%,0.369671,0.581884,0.460949,43.561436,51.146847,37.175262,0.377522,0.789810,0.312403,0.530000,0.468649,0.722683
75%,0.661670,0.696362,0.639696,64.474800,58.790279,55.525650,0.663787,0.925817,0.569056,0.845738,0.585350,0.850215
max,0.870582,0.840414,0.821933,108.440758,78.905006,98.879723,0.914793,0.990527,0.788140,0.947360,0.829983,1.000000


In [187]:
tta_con_lr5_augs = pd.read_csv(tta_con_lr5_augs_paths)
tta_con_lr5_augs = clean_df(tta_con_lr5_augs, True, 'brats')
tta_con_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.245394,0.404662,0.367935,21.784877,29.542901,27.145233,0.215324,0.362663,0.306060,0.768623,0.663402,0.735926
std,0.235258,0.274748,0.283922,17.634232,17.833599,23.112213,0.260957,0.258079,0.265048,0.257831,0.273881,0.221978
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.023743,0.167463,0.068145,12.000000,17.836666,11.678908,0.012027,0.105834,0.035680,0.748748,0.606500,0.661992
50%,0.181264,0.437390,0.424464,18.138357,27.166155,22.000000,0.100233,0.454338,0.365729,0.841615,0.727475,0.773595
75%,0.392276,0.628339,0.592325,25.610003,38.186096,29.665707,0.364506,0.546768,0.477535,0.928076,0.855239,0.865746
max,0.655798,0.859434,0.758894,80.286987,78.865067,88.674690,0.845336,0.816377,0.752396,1.000000,0.912495,0.978959


In [188]:
tta_con_lr6 = pd.read_csv(tta_con_lr6_paths)
tta_con_lr6 = clean_df(tta_con_lr6, True, 'brats')
tta_con_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.361619,0.528643,0.382829,50.395415,49.466295,38.148290,0.434228,0.774860,0.331296,0.412824,0.417254,0.631166
std,0.291667,0.178793,0.279479,29.017413,18.442688,26.970263,0.309480,0.212857,0.278837,0.346912,0.166823,0.259494
min,0.000000,0.119555,0.000000,5.873516,7.280110,4.242640,0.000000,0.155660,0.000000,0.000000,0.065731,0.000000
25%,0.068762,0.427271,0.086684,26.939736,42.693741,12.426615,0.089883,0.651947,0.047701,0.051397,0.303255,0.510653
50%,0.368173,0.572590,0.406083,50.245140,52.868402,39.044493,0.472086,0.851203,0.324342,0.358189,0.425222,0.692102
75%,0.598330,0.657138,0.631386,73.784719,60.105639,51.318416,0.694181,0.935564,0.554569,0.748586,0.512723,0.841309
max,0.872126,0.799087,0.839233,108.440750,81.185272,99.284439,0.974625,0.988613,0.847167,0.932567,0.787823,1.000000


In [190]:
tta_con_lr6_augs = pd.read_csv(tta_con_lr6_augs_paths)
tta_con_lr6_augs = clean_df(tta_con_lr6_augs, True, 'brats')
tta_con_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.250943,0.401147,0.357487,22.751818,29.832022,26.537624,0.220402,0.374665,0.296128,0.701747,0.659653,0.696601
std,0.229247,0.275123,0.302357,20.713639,15.612966,22.863047,0.260586,0.276785,0.275848,0.291632,0.257933,0.287019
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.031677,0.167472,0.036682,12.327061,20.078730,13.608030,0.016229,0.100521,0.018713,0.512277,0.562595,0.674023
50%,0.266521,0.419775,0.354955,15.762087,27.509256,20.698039,0.158382,0.420268,0.255287,0.820861,0.790975,0.772903
75%,0.432018,0.625938,0.654078,24.140080,34.151086,28.878052,0.314880,0.564264,0.539638,0.923978,0.836936,0.868485
max,0.681677,0.865077,0.778833,78.898666,74.972328,89.699501,0.876033,0.842027,0.777482,1.000000,0.889426,0.990998


In [191]:
tta_con_first_layer_lr4_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_first_layer_lr4_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_first_layer_lr5_paths =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_first_layer_lr5_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_first_layer_lr6_paths =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [192]:
tta_first_layer_lr4 = pd.read_csv(tta_con_first_layer_lr4_paths)
tta_first_layer_lr4 = clean_df(tta_first_layer_lr4, True, 'brats')
tta_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.318916,0.454354,0.342261,49.623191,51.244980,35.698439,0.385446,0.816853,0.285708,0.388281,0.330304,0.686407
std,0.287983,0.174607,0.256766,28.044760,16.247266,24.146278,0.321663,0.234559,0.247565,0.339596,0.159564,0.247729
min,0.000000,0.000000,0.000000,4.123106,7.810250,5.744563,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.046912,0.367178,0.073749,25.827575,41.309772,14.530287,0.044471,0.784815,0.040111,0.037767,0.253103,0.611998
50%,0.263435,0.470775,0.355147,51.286880,53.446167,30.853215,0.324603,0.903032,0.235806,0.322244,0.314735,0.737496
75%,0.559767,0.542400,0.587763,69.655001,61.127134,52.167602,0.688415,0.954933,0.485629,0.685047,0.410376,0.836538
max,0.849898,0.808691,0.730604,109.095375,80.529495,98.266983,0.937288,0.988304,0.855069,0.969697,0.782855,1.000000


In [193]:
tta_first_layer_lr4_augs = pd.read_csv(tta_con_first_layer_lr4_augs_paths)
tta_first_layer_lr4_augs = clean_df(tta_first_layer_lr4_augs, True, 'brats')
tta_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.220796,0.455508,0.404998,24.437978,34.562255,28.512977,0.212750,0.455637,0.328840,0.495571,0.550343,0.724766
std,0.247340,0.236267,0.261056,20.216547,18.464357,24.586843,0.278960,0.277866,0.250317,0.317069,0.238309,0.264992
min,0.000000,0.000000,0.000302,5.656854,9.110434,6.480741,0.000000,0.000000,0.000179,0.000000,0.000000,0.000949
25%,0.024242,0.359619,0.170409,13.114877,18.841444,11.785553,0.013629,0.254061,0.096450,0.273723,0.416949,0.676523
50%,0.033545,0.481215,0.416164,18.267431,32.000000,17.349352,0.061017,0.482969,0.371107,0.589474,0.572054,0.792397
75%,0.412912,0.661834,0.650820,25.495098,51.951900,30.870699,0.299788,0.694880,0.515928,0.734021,0.681968,0.903232
max,0.728504,0.786236,0.777449,78.606613,65.919647,87.409958,0.827499,0.977458,0.768060,1.000000,0.880469,0.998125


In [194]:
tta_first_layer_lr5 = pd.read_csv(tta_con_first_layer_lr5_paths)
tta_first_layer_lr5 = clean_df(tta_first_layer_lr5, True, 'brats')
tta_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.376815,0.556314,0.394697,44.241524,48.115984,38.196670,0.395716,0.742632,0.332276,0.494947,0.463210,0.689278
std,0.306446,0.187863,0.278366,28.599395,18.353861,27.983568,0.317030,0.231115,0.270068,0.366123,0.177939,0.241057
min,0.000000,0.080389,0.000000,5.000000,7.348469,4.358899,0.000000,0.067707,0.000000,0.000000,0.065890,0.000000
25%,0.074619,0.434423,0.111829,17.092392,42.154476,11.738757,0.044150,0.626871,0.059745,0.125000,0.356800,0.579162
50%,0.409832,0.580766,0.463113,45.398239,51.692337,35.581581,0.369754,0.809229,0.314321,0.546849,0.458637,0.726756
75%,0.662204,0.688956,0.650641,63.812225,58.532043,55.812183,0.643331,0.929682,0.567294,0.853971,0.572321,0.860979
max,0.874566,0.832600,0.826188,107.172760,85.915070,98.849373,0.908931,0.993251,0.786233,1.000000,0.825682,1.000000


In [195]:
tta_first_layer_lr5_augs = pd.read_csv(tta_con_first_layer_lr5_augs_paths)
tta_first_layer_lr5_augs = clean_df(tta_first_layer_lr5_augs, True, 'brats')
tta_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.240384,0.405894,0.372247,21.777053,31.450254,27.051796,0.212911,0.363629,0.312507,0.774251,0.660333,0.734611
std,0.237467,0.273433,0.284825,17.578677,21.531173,23.161337,0.266613,0.255838,0.271585,0.257132,0.271875,0.223260
min,0.000000,0.000000,0.000000,4.898980,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.023294,0.160812,0.090430,12.689544,18.381527,10.794673,0.011796,0.116809,0.048221,0.755515,0.598790,0.647321
50%,0.164674,0.427730,0.407595,19.131126,26.730129,22.202459,0.091971,0.448453,0.341310,0.845932,0.705229,0.772663
75%,0.388997,0.626057,0.592183,25.636250,38.088192,29.626902,0.361827,0.548986,0.480975,0.930490,0.853470,0.860057
max,0.658851,0.864561,0.781392,80.956779,83.812576,88.680328,0.887249,0.830647,0.768405,0.992126,0.911762,0.978932


In [196]:
tta_first_layer_lr6 = pd.read_csv(tta_con_first_layer_lr6_paths)
tta_first_layer_lr6 = clean_df(tta_first_layer_lr6, True, 'brats')
tta_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.363255,0.528842,0.382916,50.131090,49.465643,37.495580,0.435061,0.775095,0.330620,0.414470,0.417345,0.633745
std,0.292754,0.178589,0.279769,28.882287,18.469579,27.180566,0.309925,0.213148,0.278337,0.347240,0.166603,0.259829
min,0.000000,0.119538,0.000000,5.830952,7.280110,4.242640,0.000000,0.154989,0.000000,0.000000,0.065710,0.000000
25%,0.067312,0.428101,0.086108,26.903445,42.704079,12.331459,0.089427,0.650798,0.047122,0.056129,0.303416,0.512580
50%,0.368933,0.571485,0.410061,50.233370,52.887077,37.125412,0.476068,0.850753,0.322832,0.362578,0.426064,0.691532
75%,0.610011,0.656749,0.631662,69.956337,60.118154,51.044770,0.695300,0.936388,0.550629,0.750455,0.512169,0.844538
max,0.873808,0.798393,0.838867,108.288940,81.203445,99.295013,0.974625,0.989018,0.846894,0.935427,0.788549,1.000000


In [197]:
tta_first_layer_lr6_augs = pd.read_csv(tta_con_first_layer_lr6_augs_paths)
tta_first_layer_lr6_augs = clean_df(tta_first_layer_lr6_augs, True, 'brats')
tta_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.250539,0.401717,0.358595,22.776981,29.819979,26.502313,0.220452,0.375181,0.297215,0.703495,0.659173,0.697113
std,0.230150,0.275424,0.302647,20.709208,15.612593,22.868413,0.262047,0.276917,0.276104,0.289161,0.257070,0.287227
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.031758,0.167074,0.038578,12.489236,20.078730,13.590218,0.016273,0.100613,0.019707,0.517871,0.563223,0.672651
50%,0.263765,0.420274,0.356396,15.762087,27.509256,20.542119,0.156400,0.420021,0.256789,0.818623,0.785115,0.782885
75%,0.426444,0.626157,0.658964,24.122797,33.933027,28.891216,0.313791,0.565649,0.536075,0.921989,0.837084,0.869834
max,0.686622,0.865248,0.778689,78.930351,74.953316,89.699501,0.879575,0.842720,0.775535,1.000000,0.889012,0.991292


In [198]:
tta_con_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_con_norm_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_norm_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_con_norm_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_con_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_con_norm_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"


In [199]:
tta_con_norm_lr4 = pd.read_csv(tta_con_norm_lr4_paths)
tta_con_norm_lr4 = clean_df(tta_con_norm_lr4, True, 'brats')
tta_con_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.309520,0.449830,0.342946,51.661807,52.029346,35.251513,0.377771,0.812480,0.288364,0.408740,0.326579,0.664515
std,0.279550,0.170781,0.257087,25.806261,16.162368,25.076413,0.324405,0.235445,0.250673,0.346867,0.157933,0.257671
min,0.000000,0.000000,0.000000,4.472136,8.602325,5.196152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.081524,0.401541,0.093461,29.663683,43.250608,13.428240,0.057320,0.759575,0.053085,0.045380,0.256631,0.603401
50%,0.247701,0.456316,0.348194,51.738543,55.194122,32.949101,0.329706,0.900612,0.230589,0.405437,0.313348,0.689405
75%,0.556818,0.542895,0.569167,71.606947,63.101019,51.525196,0.712752,0.964472,0.522575,0.716824,0.385833,0.849877
max,0.821890,0.784804,0.757370,104.923782,80.901176,99.473358,0.942739,0.991163,0.865430,0.946878,0.783462,1.000000


In [200]:
tta_con_norm_lr4_augs = pd.read_csv(tta_con_norm_lr4_augs_paths)
tta_con_norm_lr4_augs = clean_df(tta_con_norm_lr4_augs, True, 'brats')
tta_con_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.190089,0.474423,0.412195,27.435385,35.232645,27.807267,0.178745,0.464862,0.330124,0.409011,0.563172,0.750140
std,0.239210,0.241342,0.265394,23.574001,20.775183,27.583825,0.251580,0.283305,0.249767,0.332481,0.257555,0.273428
min,0.000000,0.000000,0.000000,5.477226,9.695360,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.003012,0.343924,0.153525,14.159846,20.375099,9.524938,0.003356,0.240328,0.083651,0.070355,0.487189,0.700703
50%,0.030211,0.551916,0.413134,21.000000,32.863354,15.165751,0.017036,0.473912,0.295088,0.383611,0.577267,0.803192
75%,0.352734,0.628960,0.638720,27.930870,43.168280,32.375449,0.257351,0.679458,0.542085,0.643973,0.752900,0.935704
max,0.703095,0.791325,0.793280,106.094299,93.198715,106.317924,0.780759,0.970843,0.811208,0.994595,0.893029,0.995900


In [201]:
tta_con_norm_lr5 = pd.read_csv(tta_con_norm_lr5_paths)
tta_con_norm_lr5 = clean_df(tta_con_norm_lr5, True, 'brats')
tta_con_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.374116,0.555980,0.394030,45.198875,47.969394,38.779378,0.397349,0.739759,0.332468,0.485116,0.463963,0.682329
std,0.305433,0.189302,0.277802,28.083340,18.328452,27.710760,0.317489,0.229897,0.270561,0.363135,0.180678,0.241309
min,0.000000,0.080868,0.000000,4.358899,7.348469,4.582576,0.000000,0.067918,0.000000,0.000000,0.066196,0.000000
25%,0.074775,0.434321,0.104851,26.019224,42.359741,12.041595,0.055642,0.622531,0.058950,0.132296,0.349695,0.563502
50%,0.364814,0.587247,0.461583,43.462627,51.481071,36.345562,0.347379,0.799834,0.310197,0.483718,0.456155,0.716965
75%,0.658889,0.687077,0.640791,64.509689,58.497864,56.595516,0.661432,0.929092,0.570063,0.847970,0.571230,0.851101
max,0.873417,0.838356,0.826995,107.511864,82.716385,98.845329,0.910627,0.991909,0.789502,1.000000,0.829182,1.000000


In [202]:
tta_con_norm_lr5_augs = pd.read_csv(tta_con_norm_lr5_augs_paths)
tta_con_norm_lr5_augs = clean_df(tta_con_norm_lr5_augs, True, 'brats')
tta_con_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.239649,0.406435,0.365843,21.521985,29.519685,27.200784,0.212223,0.363387,0.300286,0.770285,0.662984,0.735950
std,0.234049,0.274408,0.281521,17.607720,17.409568,23.108416,0.264917,0.256749,0.255138,0.261003,0.270758,0.221773
min,0.000000,0.000000,0.000000,4.898980,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.030024,0.162506,0.067263,12.205456,18.444671,11.661844,0.015277,0.106841,0.035196,0.743904,0.623854,0.669994
50%,0.173780,0.449875,0.416196,18.055470,26.935104,22.000000,0.097104,0.453603,0.336382,0.845986,0.721662,0.773573
75%,0.390335,0.630336,0.589218,25.760655,38.047903,29.744385,0.351571,0.541150,0.485443,0.930254,0.854160,0.865471
max,0.647922,0.867839,0.761551,80.361679,76.242050,88.680328,0.877804,0.836221,0.681293,1.000000,0.907071,0.977423


In [203]:
tta_con_norm_lr6_augs = pd.read_csv(tta_con_norm_lr6_augs_paths)
tta_con_norm_lr6_augs = clean_df(tta_con_norm_lr6_augs, True, 'brats')
tta_con_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.250388,0.401397,0.357821,22.785560,29.824802,26.536316,0.220053,0.374963,0.296486,0.702352,0.659091,0.696566
std,0.229023,0.275174,0.302257,20.699833,15.601870,22.863179,0.260841,0.276870,0.275847,0.289804,0.257488,0.287094
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.031874,0.167289,0.037445,12.579454,20.098345,13.626971,0.016334,0.100635,0.019112,0.512564,0.562458,0.671983
50%,0.262992,0.420890,0.356175,15.762087,27.509256,20.698039,0.155899,0.420731,0.256562,0.814887,0.787730,0.776609
75%,0.428540,0.625802,0.654375,24.140080,33.790291,28.891216,0.313522,0.564878,0.537734,0.923601,0.837047,0.868864
max,0.680860,0.865216,0.778717,78.911034,74.953316,89.699501,0.876623,0.842656,0.777392,1.000000,0.889017,0.991137


In [204]:
tta_con_norm_lr6 = pd.read_csv(tta_con_norm_lr6_paths)
tta_con_norm_lr6 = clean_df(tta_con_norm_lr6, True, 'brats')
tta_con_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.361914,0.528561,0.382838,50.316004,49.477648,38.112617,0.434377,0.775200,0.331007,0.413205,0.417043,0.632163
std,0.292128,0.178589,0.279710,28.967454,18.456380,26.982631,0.309698,0.212939,0.278706,0.346946,0.166699,0.259645
min,0.000000,0.119574,0.000000,5.916080,7.280110,4.242640,0.000000,0.156426,0.000000,0.000000,0.065729,0.000000
25%,0.069182,0.427078,0.086426,26.942446,42.719440,12.350521,0.089655,0.649927,0.047437,0.054880,0.302999,0.511099
50%,0.367207,0.571921,0.408556,50.247540,52.891745,38.083952,0.469009,0.851203,0.324357,0.357573,0.425441,0.691849
75%,0.602718,0.656670,0.631484,73.068785,60.114025,51.278326,0.695656,0.936216,0.553271,0.749223,0.512129,0.842989
max,0.872988,0.798914,0.839163,108.313187,81.201904,99.298790,0.974625,0.988780,0.847167,0.933675,0.788054,1.000000


In [205]:
mask = tta_con_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.387165,0.489198,0.415400,57.728818,53.310083,38.801935,0.466671,0.845586,0.371916,0.464095,0.360487,0.635447
std,0.304132,0.143282,0.250755,23.979651,15.142735,30.137946,0.331580,0.170878,0.267507,0.344017,0.145002,0.228191
min,0.000000,0.107058,0.000000,17.832554,17.464249,5.830952,0.000000,0.203907,0.000000,0.000000,0.058325,0.000000
25%,0.075208,0.421869,0.247325,40.853397,49.416595,15.264338,0.171053,0.804810,0.172444,0.102027,0.281506,0.560568
50%,0.422640,0.460894,0.443048,57.710918,54.460995,33.630344,0.497225,0.915466,0.363751,0.590278,0.318357,0.665250
75%,0.719456,0.573968,0.639112,70.763680,62.825153,56.011604,0.780169,0.955399,0.592492,0.736007,0.409146,0.770048
max,0.797493,0.794996,0.795810,105.312630,77.362785,113.021019,0.914928,0.992901,0.829081,0.947493,0.774589,0.973783


In [206]:
mask = tta_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.392541,0.496928,0.367414,49.797669,51.707767,35.342241,0.467292,0.867646,0.319506,0.446774,0.367206,0.668533
std,0.299121,0.147612,0.253961,29.159947,17.443210,26.173081,0.326700,0.153287,0.261057,0.333652,0.152266,0.228739
min,0.000000,0.093761,0.000000,4.123106,7.810250,6.708204,0.000000,0.211837,0.000000,0.000000,0.049786,0.000000
25%,0.075152,0.447049,0.129866,26.076809,47.021271,11.874342,0.244118,0.830099,0.070575,0.131768,0.290170,0.604186
50%,0.353533,0.507668,0.372929,52.810986,56.964901,27.892651,0.439624,0.941717,0.249339,0.446157,0.352143,0.669886
75%,0.655669,0.585230,0.600394,70.146988,62.498001,51.439285,0.783841,0.965534,0.492313,0.735324,0.420798,0.788652
max,0.849898,0.808691,0.730604,109.095375,77.012985,98.266983,0.937288,0.978789,0.855069,0.960386,0.782855,1.000000


In [207]:
mask = tta_con_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.391006,0.491186,0.377373,51.977592,52.254547,34.030107,0.460500,0.856997,0.331611,0.480845,0.363838,0.657815
std,0.287314,0.144023,0.250870,26.358177,16.830557,27.022669,0.312102,0.160687,0.262157,0.335538,0.154012,0.233939
min,0.000000,0.110039,0.000000,4.472136,8.602325,5.196152,0.000000,0.215802,0.000000,0.000000,0.059602,0.000000
25%,0.103756,0.435228,0.128773,28.792360,44.598206,11.401754,0.204270,0.773354,0.070148,0.156012,0.290419,0.607573
50%,0.358362,0.492505,0.358111,53.667446,56.833088,24.515301,0.485465,0.934514,0.263979,0.553238,0.345198,0.661401
75%,0.675013,0.554987,0.606316,72.205948,63.039669,53.684261,0.769565,0.966429,0.545885,0.771464,0.424093,0.783917
max,0.821890,0.784804,0.757370,104.923782,77.269012,99.473358,0.891938,0.986303,0.865430,0.946878,0.783462,1.000000


In [208]:
mask = tta_con_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.254167,0.416669,0.332437,23.703523,29.744352,28.457199,0.226512,0.396781,0.267264,0.703034,0.656087,0.678324
std,0.210405,0.280780,0.286211,21.997431,16.841577,23.759287,0.261212,0.284309,0.252687,0.273942,0.267008,0.305334
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.058200,0.169701,0.040975,12.447567,19.052486,14.552272,0.030172,0.103408,0.020962,0.555110,0.570346,0.629725
50%,0.273313,0.505806,0.318666,15.524175,26.645824,21.000000,0.162358,0.466735,0.218011,0.817046,0.787862,0.726772
75%,0.404094,0.651915,0.625596,25.593387,38.484299,29.357861,0.295245,0.592874,0.514930,0.905596,0.833434,0.892735
max,0.627388,0.865077,0.733557,78.898666,74.972328,89.699501,0.876033,0.842027,0.651767,1.000000,0.889426,0.990998


In [209]:
mask = tta_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.253370,0.417220,0.333717,23.737075,29.731846,28.416312,0.226168,0.397340,0.268665,0.704135,0.655249,0.678571
std,0.211085,0.281128,0.286700,21.987461,16.841136,23.768793,0.262474,0.284425,0.253452,0.273103,0.266309,0.305445
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.056630,0.169473,0.043682,12.568798,19.052486,14.516648,0.029330,0.103505,0.022389,0.562934,0.569940,0.627359
50%,0.263767,0.505985,0.320447,15.524175,26.645824,20.688160,0.157045,0.476675,0.219817,0.814103,0.776609,0.722245
75%,0.398897,0.652120,0.633773,25.573714,38.387190,29.248021,0.291272,0.593232,0.517053,0.904584,0.833981,0.891800
max,0.630881,0.865248,0.734134,78.930351,74.953316,89.699501,0.879575,0.842720,0.653892,1.000000,0.889012,0.991292


In [210]:
mask = tta_con_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.253519,0.416904,0.332819,23.738240,29.730463,28.455684,0.226133,0.397048,0.267685,0.702893,0.655452,0.678137
std,0.210282,0.280831,0.286115,21.980631,16.829358,23.759565,0.261603,0.284404,0.252747,0.273453,0.266510,0.305349
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.057633,0.169617,0.041995,12.628943,19.065563,14.517834,0.029867,0.103537,0.021499,0.557790,0.569090,0.628242
50%,0.264944,0.506458,0.319166,15.524175,26.683329,21.000000,0.156558,0.467529,0.218427,0.810127,0.781919,0.724859
75%,0.401069,0.651898,0.625989,25.593387,38.283501,29.349441,0.292791,0.592446,0.512140,0.905652,0.833992,0.891803
max,0.630438,0.865216,0.734230,78.911034,74.953316,89.699501,0.876623,0.842656,0.653822,1.000000,0.889017,0.991137


In [211]:
mask = tta_con_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.349010,0.462546,0.471791,22.228237,42.282438,28.154997,0.298376,0.522885,0.429404,0.593708,0.490933,0.667268
std,0.284146,0.249277,0.268139,23.855237,21.057138,28.068838,0.287428,0.310430,0.286132,0.321166,0.245714,0.252662
min,0.000000,0.000000,0.000000,5.196152,11.357817,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.092902,0.310593,0.232020,9.675508,26.214947,9.317410,0.054088,0.337613,0.132930,0.351941,0.428045,0.586885
50%,0.368177,0.548549,0.558150,14.594520,39.874805,16.278820,0.240932,0.550439,0.530373,0.722695,0.487793,0.724853
75%,0.567573,0.649482,0.693296,20.500000,48.933584,37.036777,0.479524,0.783392,0.688178,0.880731,0.653280,0.814051
max,0.789069,0.728823,0.766090,95.905159,94.005318,107.916634,0.914116,0.947865,0.787569,0.930940,0.874882,0.991698


In [212]:
mask = tta_first_layer_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000
mean,0.282146,0.422564,0.351479,26.359309,38.204516,34.131721,0.272082,0.448603,0.273359,0.564550,0.508230,0.702380
std,0.254275,0.249448,0.258854,22.752635,19.011975,25.754617,0.296395,0.310525,0.236101,0.285418,0.242157,0.275024
min,0.001599,0.000000,0.000302,5.656854,9.110434,6.480741,0.000800,0.000000,0.000179,0.109589,0.000000,0.000949
25%,0.022585,0.205657,0.089341,12.117926,20.815227,16.500000,0.012496,0.175768,0.046958,0.320909,0.392399,0.677530
50%,0.257896,0.476914,0.291730,19.869171,39.670998,27.901439,0.175028,0.459295,0.189748,0.625286,0.537848,0.761595
75%,0.435202,0.606474,0.610450,26.983416,54.141379,42.283486,0.453915,0.706721,0.485554,0.755470,0.666206,0.878993
max,0.728504,0.786236,0.703033,78.606613,65.919647,87.409958,0.827499,0.977458,0.717246,1.000000,0.831626,0.998125


In [213]:
mask = tta_con_norm_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.275741,0.418178,0.346631,28.957280,41.919187,36.808671,0.264865,0.433062,0.267142,0.459779,0.492707,0.702987
std,0.254702,0.259534,0.260054,28.396171,22.465955,30.665447,0.275414,0.320118,0.224420,0.321788,0.280285,0.302260
min,0.000000,0.000000,0.000000,5.477226,9.695360,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.019547,0.230461,0.110227,12.400319,29.431677,16.582875,0.010748,0.140953,0.059050,0.210426,0.406751,0.700703
50%,0.249435,0.449757,0.413134,21.000000,38.483765,29.137604,0.197383,0.442142,0.290672,0.383611,0.522548,0.768454
75%,0.461400,0.602564,0.571857,27.930870,56.019745,45.285963,0.536370,0.657518,0.446873,0.661812,0.717313,0.877246
max,0.703095,0.772913,0.687086,106.094299,93.198715,106.317924,0.780759,0.970843,0.590015,0.994595,0.832151,0.995900


In [214]:
mask = tta_con_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.258557,0.425158,0.386822,21.754131,29.371155,27.095763,0.227050,0.381797,0.322273,0.755768,0.653690,0.731500
std,0.234768,0.267349,0.279602,18.144951,18.334446,23.781235,0.263321,0.251312,0.262859,0.258966,0.278435,0.227550
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.042822,0.197564,0.091389,12.000000,17.254999,11.518363,0.022271,0.147794,0.048405,0.747717,0.586230,0.653089
50%,0.204004,0.481306,0.441671,16.434639,25.093942,20.246621,0.116556,0.481980,0.365917,0.840536,0.721607,0.763672
75%,0.413173,0.634908,0.602287,25.805002,40.834579,29.886921,0.430892,0.552040,0.496507,0.901835,0.860306,0.875248
max,0.655798,0.859434,0.758894,80.286987,78.865067,88.674690,0.845336,0.816377,0.752396,1.000000,0.912495,0.978959


In [215]:
mask = tta_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.253142,0.426435,0.391279,21.753356,31.394715,26.998126,0.224440,0.382805,0.329029,0.762701,0.650638,0.730465
std,0.237555,0.265848,0.280376,18.087998,22.153994,23.831603,0.269425,0.248809,0.269454,0.259466,0.276357,0.228979
min,0.000000,0.000000,0.000000,4.898980,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.038236,0.195843,0.114719,12.344772,17.572290,10.520812,0.019726,0.163876,0.061627,0.751664,0.583444,0.637467
50%,0.166180,0.484120,0.457893,17.327651,24.994768,20.239063,0.091991,0.477862,0.358361,0.843729,0.705204,0.766828
75%,0.408357,0.636000,0.608668,25.818125,40.794816,29.931734,0.427408,0.555176,0.505720,0.909722,0.853857,0.866599
max,0.658851,0.864561,0.781392,80.956779,83.812576,88.680328,0.887249,0.830647,0.768405,0.992126,0.911762,0.978932


In [216]:
mask = tta_con_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.252525,0.427031,0.384600,21.475143,29.359462,27.154401,0.223793,0.382562,0.316171,0.757523,0.653546,0.731693
std,0.233807,0.266826,0.277198,18.116976,17.899877,23.777454,0.267611,0.249802,0.252683,0.262399,0.275373,0.227403
min,0.000000,0.000000,0.000000,4.898980,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.049264,0.189989,0.091544,12.123525,17.667006,11.376190,0.025471,0.148252,0.048487,0.736344,0.620424,0.662358
50%,0.183618,0.492796,0.450280,17.774162,25.907404,20.340771,0.102426,0.478381,0.362995,0.841009,0.715592,0.760133
75%,0.409928,0.633824,0.601006,26.004733,40.776887,30.004938,0.411708,0.544890,0.505499,0.913970,0.856417,0.874648
max,0.647922,0.867839,0.761551,80.361679,76.242050,88.680328,0.877804,0.836221,0.681293,1.000000,0.907071,0.977423


In [217]:
mask = tta_con_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.439874,0.586597,0.406333,46.391747,47.183584,37.178114,0.465585,0.777913,0.349123,0.498041,0.494326,0.662160
std,0.319163,0.173719,0.265135,30.410797,19.946544,29.362408,0.317496,0.207671,0.268245,0.370738,0.174664,0.234344
min,0.000000,0.074081,0.000565,5.000000,7.280110,4.472136,0.000000,0.061329,0.000286,0.000000,0.066741,0.021246
25%,0.075188,0.560414,0.177569,26.000000,43.703548,10.392304,0.155652,0.750201,0.102786,0.156852,0.420634,0.548757
50%,0.413777,0.627229,0.460949,44.395947,52.019226,36.359318,0.430333,0.835557,0.312403,0.570423,0.502052,0.722683
75%,0.742918,0.696362,0.638572,64.474800,59.211487,54.635155,0.782855,0.925817,0.569056,0.845738,0.585350,0.823380
max,0.870582,0.788464,0.818002,108.440758,76.909035,98.879723,0.914793,0.990527,0.788140,0.947360,0.829983,1.000000


In [218]:
mask = tta_con_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.439634,0.583884,0.408299,47.079010,47.404872,37.645889,0.466870,0.784391,0.350588,0.500709,0.488928,0.666263
std,0.319007,0.171268,0.264540,29.939989,20.087411,28.872640,0.318166,0.206954,0.268063,0.367206,0.173307,0.233368
min,0.000000,0.080868,0.000674,4.358899,7.348469,4.582576,0.000000,0.067918,0.000343,0.000000,0.066196,0.018519
25%,0.084017,0.559535,0.172795,26.396002,43.462627,11.575837,0.220404,0.761507,0.099452,0.138667,0.423030,0.563502
50%,0.404421,0.619075,0.461583,44.839153,51.874851,32.062439,0.429377,0.849949,0.310197,0.548980,0.489248,0.716965
75%,0.746773,0.687077,0.636626,64.509689,58.668560,53.768021,0.780549,0.934419,0.570063,0.845248,0.577625,0.825653
max,0.873417,0.793904,0.819145,107.511864,77.388626,98.845329,0.910627,0.991909,0.789502,0.950065,0.829182,1.000000


In [219]:
mask = tta_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.443590,0.583889,0.408891,47.361235,47.444018,37.101086,0.466227,0.788324,0.350267,0.505850,0.487241,0.672921
std,0.319873,0.171068,0.266430,29.369125,19.931156,29.345821,0.318497,0.206860,0.268347,0.371051,0.172539,0.234507
min,0.000000,0.080389,0.000487,5.000000,7.348469,4.358899,0.000000,0.067707,0.000248,0.000000,0.065890,0.013771
25%,0.080974,0.560567,0.148194,26.324873,43.566040,10.677078,0.258765,0.767863,0.082545,0.136531,0.422420,0.579162
50%,0.436915,0.615364,0.463113,45.880280,52.105663,30.800966,0.436008,0.858241,0.314321,0.546849,0.481559,0.724021
75%,0.756825,0.688956,0.638165,63.812225,58.944424,55.235859,0.783169,0.936712,0.567294,0.854879,0.576459,0.821037
max,0.874566,0.795724,0.821910,107.172760,77.110313,98.849373,0.908931,0.993251,0.786233,0.953102,0.825682,1.000000


In [220]:
mask = tta_con_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.411922,0.564715,0.401206,51.678279,48.628341,37.873973,0.460335,0.811315,0.360107,0.452646,0.450848,0.631884
std,0.314510,0.157638,0.265312,28.630841,19.382374,28.223400,0.312713,0.185368,0.280377,0.367760,0.157218,0.219818
min,0.000000,0.119555,0.004475,7.348469,7.280110,4.242640,0.000000,0.155660,0.002269,0.000000,0.065731,0.046807
25%,0.074634,0.551342,0.141455,26.981476,43.107422,10.677078,0.211328,0.763614,0.083474,0.072094,0.392126,0.530929
50%,0.371015,0.590953,0.451822,52.292442,52.211109,39.658535,0.492075,0.884375,0.363337,0.425678,0.434320,0.686234
75%,0.653057,0.665386,0.633860,73.341667,60.934391,51.072491,0.733235,0.936934,0.562385,0.864253,0.540136,0.743940
max,0.872126,0.775530,0.790928,108.440750,77.420929,99.284439,0.901783,0.988613,0.847167,0.932567,0.787823,1.000000


In [221]:
mask = tta_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.414147,0.564934,0.401325,51.508224,48.610258,36.850499,0.462297,0.811978,0.359077,0.453891,0.450943,0.634738
std,0.315526,0.157628,0.265521,28.568982,19.407915,28.516410,0.313174,0.185409,0.279632,0.367556,0.157313,0.221397
min,0.000000,0.119538,0.003988,7.280110,7.280110,4.242640,0.000000,0.154989,0.002022,0.000000,0.065710,0.045259
25%,0.074438,0.551404,0.142246,27.017569,43.127716,10.295630,0.217080,0.764022,0.084006,0.071205,0.391712,0.536293
50%,0.371734,0.595393,0.452087,52.203930,52.211109,38.420692,0.492305,0.884801,0.363427,0.433649,0.434269,0.686709
75%,0.653970,0.664375,0.634209,70.498230,60.775394,49.977909,0.731382,0.937891,0.552698,0.862338,0.539724,0.752901
max,0.873808,0.776092,0.793263,108.288940,77.420929,99.295013,0.901325,0.989018,0.846894,0.935427,0.788549,1.000000


In [222]:
mask = tta_con_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.412681,0.564608,0.401202,51.556392,48.626493,37.809229,0.461263,0.811779,0.359619,0.453037,0.450638,0.633039
std,0.315053,0.157554,0.265334,28.547554,19.394961,28.221558,0.313000,0.185243,0.280057,0.367442,0.157334,0.220729
min,0.000000,0.119574,0.004287,7.348469,7.280110,4.242640,0.000000,0.156426,0.002174,0.000000,0.065729,0.046053
25%,0.074512,0.551288,0.141875,27.053642,43.104523,10.677078,0.211564,0.763784,0.083760,0.071631,0.391552,0.532383
50%,0.370290,0.591518,0.452656,52.338322,52.211109,38.418747,0.492848,0.884740,0.364240,0.426499,0.433488,0.686161
75%,0.654076,0.664557,0.634022,70.498230,60.901562,50.912132,0.731012,0.937629,0.557657,0.863412,0.539574,0.748841
max,0.872988,0.775729,0.791700,108.313187,77.420929,99.298790,0.901572,0.988780,0.847167,0.933675,0.788054,1.000000


## ENT AND SUPERVISION

In [223]:
tta_combined_decoder_lr4_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_decoder_lr4_augs_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_decoder_lr5_paths =   "/scratch-second/TTA_results/val_combined_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_decoder_lr5_augs_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_decoder_lr6_paths =   "/scratch-second/TTA_results/val_combined_decoder_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_decoder_lr6_augs_paths  =   "/scratch-second/TTA_results/val_combined_decoder_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [224]:
tta_combined_decoder_lr6 = pd.read_csv(tta_combined_decoder_lr6_paths)
tta_combined_decoder_lr6 = clean_df(tta_combined_decoder_lr6, True, 'brats')
tta_combined_decoder_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.365312,0.526551,0.382974,49.423947,49.715768,35.847023,0.437486,0.778140,0.330776,0.415241,0.413741,0.639925
std,0.292079,0.177686,0.279657,29.507943,18.380501,26.376340,0.308062,0.211302,0.279045,0.348026,0.165262,0.254477
min,0.000000,0.119724,0.000000,6.403124,7.000000,4.000000,0.000000,0.155373,0.000000,0.000000,0.065843,0.000000
25%,0.067419,0.424644,0.082730,26.561664,42.757991,9.956723,0.125279,0.660384,0.045454,0.052383,0.305951,0.550076
50%,0.364840,0.570069,0.397519,49.889599,53.156727,33.351315,0.506418,0.847207,0.325914,0.334411,0.419244,0.691815
75%,0.621115,0.652836,0.630265,70.590389,60.373149,52.660741,0.690684,0.937139,0.563713,0.750334,0.510040,0.829024
max,0.868091,0.780894,0.841044,107.112091,81.375671,99.040398,0.967705,0.989280,0.857610,0.935544,0.772343,0.983607


In [225]:
tta_combined_decoder_lr6_augs = pd.read_csv(tta_combined_decoder_lr6_augs_paths)
tta_combined_decoder_lr6_augs = clean_df(tta_combined_decoder_lr6_augs, True, 'brats')
tta_combined_decoder_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.281953,0.422682,0.358267,22.019846,28.028094,29.359112,0.240231,0.397322,0.297313,0.661623,0.652397,0.717262
std,0.228145,0.271621,0.303169,20.299308,14.601404,24.744276,0.250963,0.273910,0.277830,0.328547,0.256939,0.277896
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044298,0.170762,0.032702,10.620908,21.373020,13.800393,0.023349,0.132745,0.016649,0.458573,0.524523,0.674275
50%,0.310973,0.441176,0.355258,14.682993,25.978251,21.141898,0.187639,0.436638,0.253659,0.768164,0.785520,0.779703
75%,0.488977,0.651566,0.629164,24.186438,31.687559,30.686098,0.356958,0.580985,0.551669,0.924808,0.821226,0.883284
max,0.676907,0.868418,0.798621,78.712128,73.545906,89.178192,0.894923,0.857119,0.788716,1.000000,0.923194,0.999000


In [226]:
tta_combined_decoder_lr5 = pd.read_csv(tta_combined_decoder_lr5_paths)
tta_combined_decoder_lr5 = clean_df(tta_combined_decoder_lr5, True, 'brats')
tta_combined_decoder_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.372859,0.539447,0.372976,47.031015,48.858694,36.814658,0.410250,0.751134,0.310889,0.456467,0.436521,0.659352
std,0.300973,0.187982,0.278315,30.317188,18.527944,26.521208,0.311495,0.225842,0.267639,0.359988,0.172812,0.259740
min,0.000000,0.096290,0.000000,5.000000,7.211102,4.123106,0.000000,0.093885,0.000000,0.000000,0.064394,0.000000
25%,0.054506,0.420627,0.075909,25.792049,42.192867,9.943642,0.090315,0.641768,0.041603,0.067752,0.329332,0.568624
50%,0.363808,0.576848,0.394853,45.464111,52.673244,35.643398,0.406124,0.827894,0.269317,0.412771,0.433372,0.705215
75%,0.626590,0.686861,0.636596,69.775263,59.342638,52.908854,0.689957,0.928211,0.541755,0.826688,0.556944,0.836131
max,0.876122,0.788624,0.830365,106.786667,80.932068,98.858482,0.966551,0.990765,0.821468,0.954415,0.812748,1.000000


In [227]:
tta_combined_decoder_lr5_augs = pd.read_csv(tta_combined_decoder_lr5_augs_paths)
tta_combined_decoder_lr5_augs = clean_df(tta_combined_decoder_lr5_augs, True, 'brats')
tta_combined_decoder_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.253326,0.397913,0.377250,24.685723,32.767792,27.919933,0.212927,0.365143,0.309818,0.643381,0.622112,0.719395
std,0.218904,0.273496,0.286893,23.403535,21.790026,25.236449,0.245254,0.268026,0.260354,0.355933,0.302518,0.270668
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.008141,0.165524,0.062808,12.165525,20.000000,12.688578,0.004087,0.112071,0.032549,0.449849,0.476352,0.681682
50%,0.275401,0.318237,0.412648,16.401220,27.018513,20.346991,0.171380,0.383504,0.316249,0.783868,0.715149,0.784437
75%,0.342271,0.642684,0.647274,26.000000,43.520111,29.495762,0.231192,0.562159,0.564214,0.918717,0.858609,0.892650
max,0.620156,0.867633,0.746059,96.900978,96.586746,107.200745,0.857143,0.843645,0.682298,1.000000,0.929865,1.000000


In [228]:
tta_combined_decoder_lr4 = pd.read_csv(tta_combined_decoder_lr4_paths)
tta_combined_decoder_lr4 = clean_df(tta_combined_decoder_lr4, True, 'brats')
tta_combined_decoder_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.325157,0.492441,0.405711,40.134944,51.684519,43.532106,0.321528,0.823941,0.364891,0.502816,0.363038,0.611572
std,0.303008,0.172619,0.269604,27.011192,14.580894,26.196930,0.301871,0.201016,0.274364,0.396216,0.147908,0.244686
min,0.000000,0.110114,0.001748,4.242640,10.770329,5.000000,0.000000,0.118081,0.000875,0.000000,0.059315,0.010140
25%,0.037702,0.378084,0.125437,17.916473,43.370499,25.238859,0.019830,0.818472,0.076152,0.041984,0.245141,0.475885
50%,0.235930,0.521771,0.492312,36.510273,53.721504,44.393131,0.249668,0.911861,0.381842,0.500000,0.374786,0.643780
75%,0.613526,0.617505,0.629977,58.905434,60.704201,58.915607,0.614100,0.946847,0.574673,0.910231,0.456859,0.763741
max,0.869218,0.793299,0.834008,106.543602,85.609581,107.122131,0.885925,0.997611,0.871251,0.997171,0.667121,1.000000


In [229]:
tta_combined_decoder_lr4_augs = pd.read_csv(tta_combined_decoder_lr4_augs_paths)
tta_combined_decoder_lr4_augs = clean_df(tta_combined_decoder_lr4_augs, True, 'brats')
tta_combined_decoder_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.246824,0.471168,0.415756,24.002027,28.405635,29.342131,0.176865,0.455639,0.347279,0.780798,0.636093,0.757085
std,0.249578,0.281383,0.281912,17.429360,16.890917,23.689843,0.196675,0.299859,0.263915,0.289106,0.246923,0.232696
min,0.000000,0.000000,0.000000,5.830952,5.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.017572,0.196320,0.172329,12.247449,14.798649,13.960634,0.008921,0.210479,0.094451,0.725000,0.554195,0.698965
50%,0.139461,0.543617,0.465059,23.402977,27.230495,20.777365,0.074957,0.433778,0.345364,0.903760,0.692397,0.809645
75%,0.386617,0.669651,0.608425,30.000000,38.755646,42.166336,0.252564,0.639931,0.538651,0.967370,0.799639,0.896829
max,0.665582,0.873493,0.854695,77.627312,72.553429,89.450546,0.556564,0.902246,0.765438,1.000000,0.862826,0.982049


In [230]:
tta_combined_first_layer_lr4_paths =    "/scratch-second/TTA_results/val_combined_first_layer_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_combined_first_layer_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_combined_first_layer_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_first_layer_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_first_layer_lr6_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_first_layer_lr6_augs_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [231]:
tta_combined_first_layer_lr4 = pd.read_csv(tta_combined_first_layer_lr4_paths)
tta_combined_first_layer_lr4 = clean_df(tta_combined_first_layer_lr4, True, 'brats')
tta_combined_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.309024,0.488056,0.410336,38.602751,51.090656,48.054601,0.295878,0.828921,0.385277,0.523814,0.358674,0.579882
std,0.289723,0.178854,0.275475,27.517177,16.436104,26.129972,0.275877,0.211935,0.292781,0.409442,0.155248,0.252364
min,0.000000,0.078737,0.000000,4.000000,12.247449,5.000000,0.000000,0.087268,0.000000,0.000000,0.059990,0.000000
25%,0.034185,0.376802,0.137475,15.811388,44.090816,33.763885,0.042021,0.825503,0.081541,0.037779,0.243580,0.425472
50%,0.196991,0.492259,0.492765,33.570801,53.188343,48.672375,0.226092,0.920360,0.425415,0.572524,0.358593,0.614612
75%,0.531248,0.609648,0.629911,55.664619,61.044617,66.001892,0.539793,0.944210,0.620647,0.945479,0.454181,0.729339
max,0.852357,0.801578,0.854009,104.686195,83.791405,107.233620,0.823197,0.996558,0.903858,1.000000,0.705058,1.000000


In [232]:
tta_combined_first_layer_lr4_augs = pd.read_csv(tta_combined_first_layer_lr4_augs_paths)
tta_combined_first_layer_lr4_augs = clean_df(tta_combined_first_layer_lr4_augs, True, 'brats')
tta_combined_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.361308,0.539836,0.464672,20.620431,28.480432,27.137251,0.276962,0.545969,0.394960,0.777860,0.594875,0.775662
std,0.266202,0.257482,0.274381,18.770225,15.727932,22.425694,0.238043,0.270295,0.265377,0.303578,0.251809,0.229227
min,0.000000,0.000000,0.000000,4.472136,5.000000,5.196152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.106151,0.356109,0.206806,8.781010,19.491737,7.953900,0.058786,0.372367,0.116045,0.736799,0.452524,0.699443
50%,0.377843,0.619870,0.570862,13.383861,23.332911,17.763135,0.257491,0.579487,0.456281,0.886183,0.701118,0.853625
75%,0.602386,0.750412,0.693431,25.500000,35.762104,40.162861,0.452436,0.762704,0.589855,0.986284,0.765311,0.920599
max,0.836873,0.875836,0.764717,78.166489,72.842293,75.610847,0.842486,0.930627,0.813177,1.000000,0.827138,0.986743


In [233]:
tta_combined_first_layer_lr5 = pd.read_csv(tta_combined_first_layer_lr5_paths)
tta_combined_first_layer_lr5 = clean_df(tta_combined_first_layer_lr5, True, 'brats')
tta_combined_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.371539,0.541256,0.378850,44.790241,48.117032,37.872793,0.401647,0.755693,0.317272,0.462276,0.437616,0.662714
std,0.303414,0.188448,0.280115,29.649265,18.050719,26.220034,0.312536,0.225232,0.270373,0.364117,0.174265,0.257382
min,0.000000,0.105763,0.000000,4.123106,7.348469,4.123106,0.000000,0.100441,0.000000,0.000000,0.063268,0.000000
25%,0.056042,0.422523,0.094676,19.869936,42.182930,10.733382,0.085939,0.644363,0.051148,0.062917,0.328974,0.578607
50%,0.358027,0.573203,0.412610,44.286648,52.523668,37.409634,0.387563,0.829776,0.291237,0.461922,0.437933,0.703909
75%,0.630250,0.684142,0.642994,68.343023,58.973597,54.436266,0.677660,0.930212,0.552667,0.843991,0.551179,0.851902
max,0.879380,0.814813,0.839447,105.638351,81.562225,98.858482,0.966551,0.991392,0.824010,0.959073,0.809140,0.988506


In [234]:
tta_combined_first_layer_lr5_augs = pd.read_csv(tta_combined_first_layer_lr5_augs_paths)
tta_combined_first_layer_lr5_augs = clean_df(tta_combined_first_layer_lr5_augs, True, 'brats')
tta_combined_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.245653,0.393741,0.366233,24.782064,34.119234,27.846364,0.209529,0.363415,0.304032,0.655635,0.618871,0.723965
std,0.228404,0.281976,0.297287,23.365823,24.447783,25.130588,0.253059,0.272381,0.271434,0.356330,0.300409,0.272512
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.007309,0.147901,0.077274,12.041595,18.411953,12.124355,0.003668,0.084332,0.040555,0.465004,0.473527,0.665307
50%,0.260915,0.318989,0.396458,17.029387,26.153393,20.024984,0.151172,0.360227,0.297983,0.855088,0.713866,0.816967
75%,0.338872,0.660779,0.694691,25.647604,43.737854,28.530685,0.224779,0.570900,0.545406,0.913193,0.862055,0.888889
max,0.632709,0.866486,0.739218,96.900978,97.015465,107.373184,0.864817,0.840498,0.726791,1.000000,0.929991,1.000000


In [235]:
tta_combined_first_layer_lr6 = pd.read_csv(tta_combined_first_layer_lr6_paths)
tta_combined_first_layer_lr6 = clean_df(tta_combined_first_layer_lr6, True, 'brats')
tta_combined_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.366268,0.526959,0.382454,49.391184,49.683343,35.826280,0.437570,0.778167,0.329752,0.416135,0.414197,0.641596
std,0.292741,0.177790,0.279695,29.506314,18.406131,26.389238,0.308248,0.211420,0.278702,0.348417,0.165426,0.254533
min,0.000000,0.119554,0.000000,6.403124,7.000000,4.036930,0.000000,0.154472,0.000000,0.000000,0.065739,0.000000
25%,0.065703,0.425475,0.082678,26.561521,42.755658,9.837473,0.125027,0.659978,0.045244,0.052573,0.306636,0.553553
50%,0.365924,0.570746,0.396242,49.658201,53.189589,33.288541,0.505789,0.847035,0.323545,0.336553,0.419372,0.694625
75%,0.623736,0.652835,0.630028,70.579948,60.261339,52.640224,0.693667,0.937234,0.563243,0.750943,0.509951,0.829613
max,0.869750,0.780783,0.840577,107.016815,81.363083,98.996971,0.967705,0.989352,0.857610,0.935591,0.774303,0.983333


In [236]:
tta_combined_first_layer_lr6_augs = pd.read_csv(tta_combined_first_layer_lr6_augs_paths)
tta_combined_first_layer_lr6_augs = clean_df(tta_combined_first_layer_lr6_augs, True, 'brats')
tta_combined_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.282657,0.423556,0.357730,21.936714,28.028338,29.373474,0.241000,0.397902,0.296472,0.662362,0.652564,0.719582
std,0.228618,0.271112,0.302892,20.281844,14.597789,24.753152,0.251313,0.273377,0.277141,0.327926,0.256850,0.278544
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044907,0.170843,0.033008,10.670113,21.359132,13.781708,0.023688,0.131966,0.016808,0.461664,0.524433,0.675095
50%,0.312050,0.440761,0.353636,14.670832,25.950019,21.153638,0.188483,0.436430,0.252594,0.770417,0.785816,0.785360
75%,0.489449,0.651583,0.628939,23.455127,31.681627,30.686871,0.357757,0.581468,0.550987,0.924502,0.819884,0.884381
max,0.679724,0.868247,0.798454,78.625687,73.545906,89.178192,0.895514,0.856425,0.786411,1.000000,0.923289,0.999004


In [237]:
tta_combined_norm_lr4_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_norm_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_norm_lr5_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_norm_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_norm_lr6_paths =  "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr6/logs/BraTS_GLI/SSA_boxes.csv"
tta_combined_norm_lr6_augs_paths =  "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr6_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [238]:
tta_combined_norm_lr4 = pd.read_csv(tta_combined_norm_lr4_paths)
tta_combined_norm_lr4 = clean_df(tta_combined_norm_lr4, True, 'brats')
tta_combined_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.295357,0.499199,0.393516,39.483140,50.462109,45.763026,0.286495,0.825200,0.363984,0.531010,0.372223,0.590637
std,0.291148,0.182104,0.277415,26.600488,16.437848,26.778080,0.279755,0.206386,0.288044,0.409335,0.162298,0.258000
min,0.000000,0.099957,0.000000,3.741657,10.246951,5.000000,0.000000,0.100939,0.000000,0.000000,0.058730,0.000000
25%,0.037708,0.371100,0.098596,17.392101,44.267715,23.621995,0.023227,0.812654,0.056412,0.038725,0.243313,0.443325
50%,0.181233,0.516804,0.486034,35.320896,53.197731,47.946077,0.200581,0.899498,0.412554,0.587045,0.395933,0.603366
75%,0.530396,0.620187,0.625362,56.402881,60.417285,60.828294,0.508102,0.949136,0.606945,0.931006,0.460637,0.775230
max,0.838169,0.810480,0.846266,105.990784,83.738281,106.812920,0.835473,0.995864,0.892915,1.000000,0.719705,1.000000


In [239]:
tta_combined_norm_lr4_augs = pd.read_csv(tta_combined_norm_lr4_augs_paths)
tta_combined_norm_lr4_augs = clean_df(tta_combined_norm_lr4_augs, True, 'brats')
tta_combined_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.323923,0.524841,0.438216,21.660313,27.558924,27.555679,0.250296,0.511481,0.366464,0.730626,0.642682,0.799642
std,0.265222,0.293677,0.282605,18.421616,15.870702,21.965564,0.243240,0.309418,0.269520,0.340255,0.237278,0.234610
min,0.000000,0.000000,0.000000,4.260068,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.032098,0.248380,0.157586,8.985280,19.815340,9.384577,0.016343,0.216073,0.086540,0.700226,0.588184,0.706498
50%,0.318663,0.642127,0.550981,15.889499,26.546217,21.027735,0.199370,0.591835,0.421418,0.847432,0.744675,0.887191
75%,0.567841,0.759717,0.683711,26.588874,32.014253,36.649884,0.408601,0.760158,0.592661,0.944638,0.787219,0.950847
max,0.837134,0.873915,0.818095,81.829094,72.000000,76.642029,0.842486,0.918400,0.798272,1.000000,0.851910,1.000000


In [240]:
tta_combined_norm_lr5 = pd.read_csv(tta_combined_norm_lr5_paths)
tta_combined_norm_lr5 = clean_df(tta_combined_norm_lr5, True, 'brats')
tta_combined_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.369521,0.539960,0.377421,46.479174,48.604338,36.987733,0.403379,0.753708,0.316427,0.455671,0.436871,0.659543
std,0.302402,0.187581,0.279675,31.096824,18.670353,26.546564,0.311066,0.225549,0.270542,0.365423,0.173363,0.257655
min,0.000000,0.099813,0.000000,4.242640,7.280110,4.123106,0.000000,0.098200,0.000000,0.000000,0.063679,0.000000
25%,0.054800,0.425727,0.089254,20.145461,42.226719,10.873831,0.086900,0.644455,0.048564,0.063434,0.328623,0.587987
50%,0.348425,0.575753,0.407815,45.588005,52.315214,35.323948,0.392421,0.830037,0.282773,0.403519,0.435612,0.702977
75%,0.626249,0.684663,0.641385,70.565788,59.221867,53.677037,0.674580,0.929710,0.549630,0.844370,0.552610,0.837280
max,0.878643,0.790413,0.830280,108.537849,80.999069,98.858482,0.966551,0.990852,0.823193,0.960046,0.811669,1.000000


In [241]:
tta_combined_norm_lr5_augs = pd.read_csv(tta_combined_norm_lr5_augs_paths)
tta_combined_norm_lr5_augs = clean_df(tta_combined_norm_lr5_augs, True, 'brats')
tta_combined_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.247681,0.404958,0.368682,23.885041,29.787330,29.697319,0.209558,0.370758,0.305295,0.683965,0.658033,0.723791
std,0.221803,0.268372,0.292171,20.532533,16.459805,26.182437,0.248393,0.266463,0.267631,0.321654,0.267623,0.273542
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.053690,0.166526,0.062206,12.165525,18.681541,12.083046,0.027946,0.095101,0.032232,0.621951,0.552926,0.663690
50%,0.271638,0.314128,0.406487,16.401220,27.000000,20.322401,0.164376,0.378569,0.310006,0.839548,0.724403,0.790512
75%,0.338066,0.660315,0.690440,26.332487,39.293026,29.257477,0.224674,0.569384,0.555269,0.917065,0.858122,0.890411
max,0.635658,0.866124,0.739471,77.478378,73.938148,89.178192,0.855372,0.840742,0.724096,1.000000,0.928227,0.993590


In [242]:
tta_combined_norm_lr6 = pd.read_csv(tta_combined_norm_lr6_paths)
tta_combined_norm_lr6 = clean_df(tta_combined_norm_lr6, True, 'brats')
tta_combined_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.365481,0.526724,0.382940,49.435197,49.689223,35.828440,0.437239,0.778063,0.330631,0.415536,0.413971,0.640482
std,0.292420,0.177661,0.279711,29.506702,18.403976,26.395945,0.308063,0.211364,0.279083,0.348430,0.165299,0.254402
min,0.000000,0.119693,0.000000,6.403124,7.000000,4.000000,0.000000,0.154779,0.000000,0.000000,0.065819,0.000000
25%,0.067526,0.424911,0.082921,26.561798,42.760320,9.911325,0.124571,0.660512,0.045470,0.052225,0.306312,0.551207
50%,0.363876,0.570785,0.397717,49.865259,53.166134,33.295633,0.503089,0.847192,0.325171,0.335414,0.419159,0.691822
75%,0.621047,0.652645,0.629803,70.590389,60.321349,52.620871,0.692574,0.937308,0.564916,0.752438,0.510007,0.829396
max,0.868769,0.780969,0.841061,107.011208,81.351089,99.040398,0.967705,0.989233,0.857791,0.935579,0.773909,0.983607


In [243]:
tta_combined_norm_lr6_augs = pd.read_csv(tta_combined_norm_lr6_augs_paths)
tta_combined_norm_lr6_augs = clean_df(tta_combined_norm_lr6_augs, True, 'brats')
tta_combined_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.288284,0.426076,0.371104,21.587225,27.498491,28.566834,0.247039,0.414084,0.306787,0.660894,0.649401,0.734982
std,0.235755,0.271486,0.296260,20.245935,15.018235,24.950529,0.255596,0.283110,0.272008,0.327985,0.260649,0.259130
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044597,0.170886,0.032762,10.669421,20.267847,13.322230,0.023516,0.133350,0.016681,0.460347,0.524379,0.674085
50%,0.311562,0.480434,0.380904,13.190034,25.965346,20.208244,0.188153,0.454582,0.279634,0.768779,0.785956,0.785216
75%,0.487845,0.651329,0.629085,23.874194,31.712470,29.339070,0.400571,0.618030,0.552325,0.922592,0.819619,0.883745
max,0.678307,0.868212,0.798490,78.676552,73.545906,89.178192,0.894333,0.856207,0.788134,1.000000,0.923289,0.998987


In [244]:
mask = tta_combined_decoder_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.416005,0.521648,0.419629,37.024925,51.835871,43.373852,0.397578,0.858827,0.386700,0.578098,0.386341,0.607180
std,0.315700,0.152338,0.255108,27.245997,15.237074,27.992788,0.299131,0.174467,0.274525,0.400427,0.131261,0.218306
min,0.000000,0.110114,0.001748,4.242640,10.770329,5.000000,0.000000,0.118081,0.000875,0.000000,0.059315,0.010140
25%,0.088477,0.464674,0.157713,14.210917,45.497253,25.238859,0.120438,0.846419,0.096158,0.148021,0.325439,0.521668
50%,0.402149,0.555133,0.492312,30.405582,53.721504,43.381397,0.364985,0.929103,0.404783,0.767720,0.397002,0.643780
75%,0.712428,0.642578,0.627232,57.706154,61.943523,58.114532,0.651039,0.946847,0.557299,0.932419,0.500826,0.739060
max,0.869218,0.734872,0.785430,106.543602,76.295479,107.122131,0.885925,0.992345,0.871251,0.997171,0.602910,1.000000


In [245]:
mask = tta_combined_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.384981,0.532028,0.421095,34.969242,50.992576,45.787713,0.347746,0.875255,0.409682,0.572054,0.395279,0.596178
std,0.309322,0.156403,0.257512,26.734129,16.332979,27.481232,0.283782,0.168908,0.293961,0.411137,0.139032,0.225768
min,0.000000,0.107828,0.003890,4.000000,12.247449,5.000000,0.000000,0.098889,0.002155,0.000000,0.059990,0.019933
25%,0.065747,0.460743,0.178779,9.386126,45.442272,23.345236,0.056131,0.828654,0.107431,0.113419,0.320356,0.508665
50%,0.375293,0.564504,0.492765,29.573582,53.376026,48.062458,0.335541,0.935259,0.425415,0.779783,0.400863,0.614612
75%,0.620560,0.623490,0.629911,49.497475,61.684681,58.932163,0.571369,0.953850,0.627317,0.945540,0.465027,0.698936
max,0.852357,0.801578,0.803013,97.812317,76.243034,107.233620,0.823197,0.994299,0.903858,0.997074,0.705058,1.000000


In [246]:
mask = tta_combined_decoder_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,0.261426,0.434711,0.390189,23.552140,31.573534,33.898182,0.187553,0.421657,0.326415,0.806141,0.606461,0.750690
std,0.242644,0.283737,0.288638,18.323127,16.925209,24.090939,0.194482,0.288585,0.268622,0.245635,0.264399,0.238599
min,0.000000,0.000000,0.000000,5.830952,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.023354,0.189010,0.152578,12.247449,21.354156,17.117243,0.011842,0.117118,0.083382,0.790336,0.493163,0.698965
50%,0.217124,0.476172,0.465059,23.402977,30.000000,28.930952,0.122286,0.433778,0.306754,0.903760,0.681080,0.809645
75%,0.386617,0.647127,0.608425,30.000000,41.988094,44.395947,0.252564,0.614205,0.538651,0.959677,0.796949,0.893571
max,0.665582,0.873493,0.754604,77.627312,72.553429,89.450546,0.556564,0.884426,0.737612,1.000000,0.862826,0.982049


In [247]:
mask = tta_combined_first_layer_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000,16.000000
mean,0.385784,0.521573,0.438153,21.819357,30.365693,31.413866,0.302416,0.526395,0.366439,0.814577,0.581759,0.764049
std,0.268593,0.263584,0.282989,20.618439,16.394092,23.177051,0.246514,0.269114,0.262059,0.255305,0.254456,0.248625
min,0.000000,0.000000,0.000000,4.472136,5.000000,5.196152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.135525,0.356109,0.135859,7.550767,20.116571,14.657207,0.072945,0.357766,0.073423,0.802352,0.452524,0.696985
50%,0.415292,0.619870,0.563770,13.383861,27.393245,26.711113,0.273059,0.579487,0.425553,0.886183,0.671542,0.853625
75%,0.602386,0.706692,0.660320,25.500000,37.347198,47.910128,0.478038,0.693355,0.589855,0.968428,0.756450,0.920599
max,0.836873,0.875836,0.764717,78.166489,72.842293,75.610847,0.842486,0.930627,0.780612,1.000000,0.827138,0.986743


In [248]:
mask = tta_combined_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.429393,0.577655,0.398508,47.442083,47.657194,38.271161,0.445048,0.802471,0.342782,0.485256,0.469925,0.662380
std,0.329554,0.164777,0.266982,30.020125,19.910297,28.121975,0.320286,0.195870,0.270724,0.372478,0.161364,0.232662
min,0.000000,0.105763,0.000075,4.123106,7.348469,4.123106,0.000000,0.100441,0.000038,0.000000,0.063268,0.001986
25%,0.065515,0.558156,0.158842,26.424419,42.000000,9.868955,0.220551,0.748286,0.091594,0.122449,0.405141,0.592366
50%,0.418813,0.611892,0.438613,45.803928,52.009613,38.017097,0.457710,0.870956,0.349507,0.524886,0.461647,0.696635
75%,0.724651,0.682433,0.643101,69.634758,59.771637,52.636013,0.758014,0.933303,0.575930,0.854651,0.551985,0.816515
max,0.879380,0.789908,0.839447,105.638351,77.388626,98.858482,0.905583,0.991392,0.824010,0.959073,0.809140,0.988506


In [249]:
mask = tta_combined_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.271152,0.414632,0.364851,24.923476,34.603339,28.729636,0.231399,0.386539,0.302055,0.719602,0.611926,0.701401
std,0.225293,0.286019,0.289922,24.614229,25.719283,26.174947,0.256618,0.274313,0.266559,0.308987,0.305060,0.276493
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.073396,0.156530,0.077366,11.264842,18.205976,12.936041,0.038249,0.132560,0.040604,0.663711,0.515282,0.650948
50%,0.269360,0.479661,0.396458,13.601471,25.729361,20.024984,0.166070,0.475229,0.297983,0.864401,0.713866,0.770486
75%,0.411305,0.660784,0.648849,25.738202,45.262302,30.765343,0.336364,0.573768,0.535455,0.924453,0.844405,0.870496
max,0.632709,0.866486,0.739218,96.900978,97.015465,107.373184,0.864817,0.840498,0.726791,1.000000,0.926809,0.966988


In [250]:
mask = tta_combined_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.389075,0.535043,0.412255,39.272032,50.714845,44.890624,0.348761,0.869797,0.391561,0.593087,0.401270,0.594011
std,0.304365,0.160530,0.252332,28.964518,16.437142,28.376897,0.276467,0.167684,0.281588,0.396387,0.148135,0.227940
min,0.000000,0.104740,0.003522,3.741657,10.246951,5.000000,0.000000,0.100939,0.002022,0.000000,0.058730,0.013662
25%,0.069180,0.460552,0.182753,15.842979,44.068130,21.023796,0.108311,0.835365,0.107216,0.227008,0.317564,0.498230
50%,0.311105,0.562927,0.487026,33.994843,53.235325,48.205288,0.350096,0.925154,0.414948,0.848341,0.404162,0.605101
75%,0.630251,0.621576,0.623863,56.822086,61.354313,59.110065,0.576052,0.949749,0.607017,0.931629,0.461747,0.704351
max,0.838169,0.810480,0.757812,105.990784,76.609398,106.812920,0.816429,0.991766,0.892915,0.999796,0.719705,1.000000


In [251]:
mask = tta_combined_norm_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.362709,0.497573,0.418094,21.355870,31.579437,32.758593,0.288725,0.483683,0.345132,0.798313,0.609199,0.780549
std,0.268869,0.288717,0.282942,19.875865,15.726424,22.848354,0.256514,0.298141,0.257735,0.255401,0.259729,0.261663
min,0.000000,0.000000,0.000000,4.260068,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.112339,0.229082,0.147453,7.569074,22.173495,17.645808,0.061240,0.197841,0.080346,0.786154,0.544225,0.706006
50%,0.352326,0.641108,0.550457,15.165751,30.000000,29.017237,0.233295,0.582944,0.402500,0.855333,0.742690,0.876261
75%,0.569303,0.714521,0.645888,26.245132,35.798031,45.354769,0.442464,0.707956,0.571486,0.936282,0.784348,0.939933
max,0.837134,0.873915,0.733487,81.829094,72.000000,76.642029,0.842486,0.918400,0.692448,1.000000,0.833541,1.000000


In [252]:
mask = tta_combined_decoder_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.429836,0.576511,0.392935,48.543772,47.994880,36.693259,0.451388,0.797225,0.337436,0.483030,0.469814,0.657671
std,0.326161,0.165011,0.266210,28.955876,19.733036,28.735384,0.317453,0.196225,0.269389,0.367521,0.161184,0.235411
min,0.000000,0.101049,0.000150,5.000000,7.211102,4.123106,0.000000,0.101705,0.000076,0.000000,0.064394,0.005413
25%,0.064575,0.556102,0.164452,26.942530,42.000000,8.062258,0.262420,0.735407,0.101560,0.071361,0.407210,0.564655
50%,0.421298,0.612413,0.434113,45.585087,53.327263,37.883980,0.472393,0.862580,0.357184,0.505802,0.452634,0.695933
75%,0.695113,0.685682,0.636271,68.209236,60.016663,51.591667,0.758705,0.929363,0.574617,0.851170,0.557255,0.816968
max,0.876122,0.788624,0.830365,106.786667,77.380882,98.858482,0.902582,0.990765,0.821468,0.954415,0.812748,1.000000


In [253]:
mask = tta_combined_decoder_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.279787,0.419843,0.376832,24.839342,33.002268,28.810949,0.235235,0.388509,0.307905,0.707964,0.617632,0.696891
std,0.213267,0.276163,0.277921,24.655209,22.954729,26.284976,0.247623,0.269327,0.253330,0.307309,0.306022,0.274395
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.112173,0.181220,0.099787,11.870681,19.669539,13.218153,0.060124,0.121473,0.053502,0.643756,0.524797,0.661619
50%,0.276002,0.499271,0.412648,13.076696,25.785662,20.346991,0.172319,0.473376,0.316249,0.842355,0.715149,0.762326
75%,0.416696,0.649561,0.624489,26.152946,45.046318,30.747881,0.322787,0.564692,0.550446,0.927359,0.840463,0.865642
max,0.620156,0.867633,0.746059,96.900978,96.586746,107.200745,0.857143,0.843645,0.682298,1.000000,0.911782,0.960139


In [254]:
mask = tta_combined_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.427246,0.577181,0.396994,47.633170,47.556683,36.880082,0.445667,0.799734,0.342033,0.481295,0.470629,0.658895
std,0.328333,0.164613,0.265864,30.280599,19.942951,28.579973,0.318496,0.195968,0.270616,0.373249,0.162012,0.232843
min,0.000000,0.104367,0.000112,4.242640,7.280110,4.123106,0.000000,0.098200,0.000057,0.000000,0.063679,0.003165
25%,0.065096,0.558371,0.167341,26.532999,42.000000,9.000000,0.254179,0.739615,0.097279,0.114094,0.406018,0.585684
50%,0.411179,0.610469,0.440468,45.877022,51.971146,36.069378,0.459321,0.866729,0.354826,0.505248,0.456907,0.695049
75%,0.715733,0.683972,0.641827,70.604530,59.999996,51.573238,0.760248,0.932673,0.575930,0.854904,0.554717,0.822571
max,0.878643,0.786728,0.830280,108.537849,77.336922,98.858482,0.903739,0.990852,0.823193,0.960046,0.811669,1.000000


In [255]:
mask = tta_combined_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.429393,0.577655,0.398508,47.442083,47.657194,38.271161,0.445048,0.802471,0.342782,0.485256,0.469925,0.662380
std,0.329554,0.164777,0.266982,30.020125,19.910297,28.121975,0.320286,0.195870,0.270724,0.372478,0.161364,0.232662
min,0.000000,0.105763,0.000075,4.123106,7.348469,4.123106,0.000000,0.100441,0.000038,0.000000,0.063268,0.001986
25%,0.065515,0.558156,0.158842,26.424419,42.000000,9.868955,0.220551,0.748286,0.091594,0.122449,0.405141,0.592366
50%,0.418813,0.611892,0.438613,45.803928,52.009613,38.017097,0.457710,0.870956,0.349507,0.524886,0.461647,0.696635
75%,0.724651,0.682433,0.643101,69.634758,59.771637,52.636013,0.758014,0.933303,0.575930,0.854651,0.551985,0.816515
max,0.879380,0.789908,0.839447,105.638351,77.388626,98.858482,0.905583,0.991392,0.824010,0.959073,0.809140,0.988506


In [256]:
mask = tta_combined_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.273393,0.425625,0.366668,23.895983,29.865722,30.775428,0.231431,0.393670,0.302852,0.750621,0.656446,0.701777
std,0.217565,0.272130,0.285161,21.626443,17.348190,27.218799,0.251498,0.268320,0.262586,0.255785,0.268925,0.278135
min,0.000000,0.000000,0.000000,5.385165,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.080729,0.168585,0.071683,11.605443,18.340771,12.842258,0.042228,0.120738,0.037504,0.681910,0.573011,0.648139
50%,0.280433,0.495966,0.406487,13.661976,25.495098,20.322401,0.173765,0.474145,0.310006,0.860106,0.724403,0.766060
75%,0.414751,0.666261,0.646599,25.738202,41.617508,36.029079,0.321916,0.575420,0.544541,0.917296,0.843910,0.889326
max,0.635658,0.866124,0.739471,77.478378,73.938148,89.178192,0.855372,0.840742,0.724096,1.000000,0.913764,0.964838


In [257]:
mask = tta_combined_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.409770,0.562788,0.400850,51.596274,49.029335,36.467323,0.459371,0.813240,0.360391,0.448851,0.447746,0.632892
std,0.316100,0.156273,0.268000,28.307937,19.121845,28.654835,0.315047,0.184217,0.282935,0.363236,0.155165,0.223993
min,0.000000,0.119693,0.004101,8.062258,7.280110,4.000000,0.000000,0.154779,0.002098,0.000000,0.065819,0.043572
25%,0.073200,0.550002,0.124729,27.147743,43.289722,9.000000,0.205039,0.741893,0.072572,0.077916,0.392594,0.550891
50%,0.399042,0.592108,0.458306,52.033638,52.564247,37.552631,0.505325,0.885044,0.343839,0.412896,0.430335,0.687163
75%,0.656216,0.661872,0.634091,71.049629,60.489670,52.819504,0.723601,0.938416,0.610503,0.818732,0.538475,0.760364
max,0.868769,0.757563,0.806145,107.011208,77.749596,99.040398,0.902770,0.989233,0.857791,0.935579,0.773909,0.983607


In [258]:
mask = tta_combined_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.294314,0.459537,0.362640,22.669800,27.886513,31.645257,0.257367,0.440771,0.291265,0.685303,0.649588,0.719492
std,0.204773,0.277415,0.282643,22.424408,16.395297,26.874892,0.255135,0.279969,0.247828,0.276287,0.263541,0.282517
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.110083,0.209969,0.029186,10.113871,19.511898,14.123105,0.058995,0.147196,0.014819,0.484528,0.557241,0.671057
50%,0.311660,0.552202,0.397173,13.038404,23.000000,21.000000,0.189749,0.501195,0.297508,0.767051,0.780075,0.790390
75%,0.475675,0.681400,0.614255,25.183321,30.476545,42.976427,0.366595,0.638518,0.526992,0.905605,0.813306,0.891357
max,0.637443,0.868212,0.735501,78.676552,73.545906,89.178192,0.894333,0.856207,0.658497,0.981393,0.886274,0.998987


In [259]:
mask = tta_combined_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.411004,0.563067,0.400412,51.539212,49.023332,36.476056,0.460192,0.813468,0.359399,0.449598,0.448003,0.634009
std,0.316591,0.156498,0.268070,28.308641,19.123162,28.661629,0.315252,0.184245,0.282563,0.363368,0.155390,0.224265
min,0.000000,0.119554,0.003878,7.874008,7.280110,4.036930,0.000000,0.154472,0.001983,0.000000,0.065739,0.043336
25%,0.073248,0.550247,0.124289,27.097958,43.294865,8.774964,0.208907,0.743218,0.072253,0.077901,0.392632,0.552935
50%,0.400690,0.592109,0.457343,52.050457,52.583267,37.549965,0.507318,0.885514,0.341237,0.417996,0.430219,0.688886
75%,0.657896,0.661795,0.634367,71.029213,60.415230,52.828011,0.723231,0.938401,0.608753,0.821784,0.538523,0.761797
max,0.869750,0.758442,0.806869,107.016815,77.736732,98.996971,0.902735,0.989352,0.857610,0.935591,0.774303,0.983333


In [260]:
mask = tta_combined_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.294503,0.460443,0.362013,22.754853,27.878984,31.658504,0.257765,0.441443,0.290451,0.685518,0.649425,0.719847
std,0.205298,0.276692,0.282314,22.563867,16.400852,26.886174,0.255559,0.279229,0.247346,0.276215,0.263564,0.282603
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.108700,0.208532,0.029563,10.115255,19.511898,14.140544,0.058223,0.153612,0.015014,0.485621,0.557757,0.671991
50%,0.312754,0.552387,0.395694,13.038404,23.000000,21.000000,0.190623,0.501159,0.295905,0.770080,0.780143,0.789831
75%,0.475723,0.681363,0.614134,24.879807,30.476545,42.976427,0.368286,0.638439,0.525316,0.904694,0.813291,0.891560
max,0.638286,0.868247,0.735109,78.625687,73.545906,89.178192,0.895514,0.856425,0.657859,0.980947,0.885838,0.999004


In [261]:
mask = tta_combined_decoder_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.409479,0.562585,0.400854,51.592126,49.065821,36.486452,0.459341,0.813221,0.360635,0.448660,0.447477,0.632291
std,0.315762,0.156278,0.268090,28.304791,19.082518,28.652667,0.314915,0.184169,0.283011,0.362893,0.155078,0.223832
min,0.000000,0.119724,0.004213,8.062258,7.280110,4.000000,0.000000,0.155373,0.002155,0.000000,0.065843,0.043929
25%,0.073316,0.549543,0.124644,27.214884,43.289722,9.055386,0.205240,0.742335,0.072523,0.077869,0.392008,0.549505
50%,0.398026,0.591039,0.458956,52.115738,52.554733,37.576588,0.505979,0.885545,0.345454,0.413331,0.429171,0.686960
75%,0.656758,0.661703,0.634660,71.049629,60.539242,52.872013,0.725454,0.938279,0.610503,0.817082,0.537402,0.761367
max,0.868091,0.757042,0.806628,107.112091,77.749596,99.040398,0.902770,0.989280,0.857610,0.935544,0.772343,0.983607


In [262]:
mask = tta_combined_decoder_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.293868,0.459401,0.362612,22.769954,27.878260,31.640525,0.257066,0.440702,0.291294,0.685000,0.649351,0.717030
std,0.204938,0.277448,0.282676,22.587417,16.404860,26.877362,0.255398,0.280008,0.248006,0.276060,0.263428,0.281738
min,0.000000,0.000000,0.000000,3.741657,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.110392,0.209284,0.029310,10.016845,19.504764,14.140544,0.059167,0.146835,0.014883,0.483623,0.557913,0.671520
50%,0.311201,0.552331,0.397375,13.038404,23.000000,21.000000,0.189050,0.501048,0.297757,0.766117,0.779648,0.780366
75%,0.476345,0.680951,0.614084,25.183321,30.476545,42.976427,0.367716,0.637894,0.527184,0.904817,0.814357,0.891191
max,0.636869,0.868418,0.736447,78.712128,73.545906,89.178192,0.894923,0.857119,0.660622,0.979891,0.886225,0.999000


In [263]:
tta_ent_sup_lr4_90 = pd.read_csv(tta_ent_sup_lr4_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_90.describe()

NameError: name 'tta_ent_sup_lr4_90_paths' is not defined

In [ ]:
tta_ent_sup_lr4_augs_90 = pd.read_csv(tta_ent_sup_lr4_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_90.describe()

In [ ]:
tta_ent_sup_lr4_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,42.000000,43.000000,42.000000,44.000000,44.000000,43.000000,42.000000,43.000000
mean,21.500000,0.324426,0.497125,0.408297,32.355058,51.194817,56.864578,0.294769,0.781463,0.407793,0.595981,0.402535,0.569422
std,12.845233,0.308138,0.191550,0.251037,32.243460,15.291997,25.379832,0.297514,0.271435,0.304682,0.419757,0.149558,0.280593
min,0.000000,0.000000,0.000000,0.000000,3.605551,9.000000,5.099020,0.000000,0.000000,0.000000,0.000000,0.035061,0.000000
25%,10.750000,0.015720,0.422997,0.213118,8.602325,42.222553,42.430933,0.013383,0.679297,0.130416,0.046548,0.313816,0.447766
50%,21.500000,0.246090,0.550298,0.436968,17.888544,53.953918,58.525200,0.226739,0.903836,0.390697,0.825866,0.413747,0.566102
75%,32.250000,0.566836,0.629583,0.612216,51.734890,58.578131,70.178596,0.487501,0.962552,0.682605,0.951014,0.487892,0.776780
max,43.000000,0.879409,0.752073,0.814000,140.143677,89.044930,105.223564,0.942033,0.991484,0.967020,1.000000,0.773544,0.993976


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,20.000000,0.495372,0.552224,0.374596,34.682193,41.166470,38.642105,0.496585,0.648930,0.301344,0.574426,0.520424,0.705790
std,11.979149,0.287805,0.203896,0.255281,24.077199,15.180727,23.014303,0.320981,0.228718,0.248674,0.323001,0.212092,0.232278
min,0.000000,0.000000,0.128629,0.012580,2.828427,5.385165,4.898980,0.000000,0.099212,0.006342,0.000000,0.106339,0.047648
25%,10.000000,0.275319,0.376829,0.113588,10.168473,32.341923,19.709137,0.229131,0.482100,0.092131,0.409091,0.343021,0.592982
50%,20.000000,0.526514,0.599774,0.384361,35.589325,43.688663,42.953465,0.491021,0.714876,0.242304,0.645089,0.584136,0.769912
75%,30.000000,0.753613,0.712928,0.563877,51.253590,53.795910,53.361034,0.829358,0.825936,0.431440,0.808208,0.686962,0.876651
max,40.000000,0.877228,0.877293,0.865718,90.262360,63.765194,88.441513,0.934045,0.960371,0.853518,0.976514,0.844974,0.993000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,37.000000,39.000000,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000
mean,19.000000,0.386572,0.542965,0.365341,38.783502,45.204024,40.426638,0.394172,0.640431,0.302453,0.558596,0.510062,0.634425
std,11.401754,0.304116,0.231474,0.274624,27.906069,18.382322,24.707967,0.326880,0.273146,0.260121,0.350189,0.219291,0.286275
min,0.000000,0.000000,0.009322,0.000038,3.743551,5.000000,2.236068,0.000000,0.006224,0.000019,0.000000,0.018557,0.000989
25%,9.500000,0.102245,0.361455,0.110163,11.357817,39.974083,18.917438,0.060110,0.435871,0.059939,0.235156,0.342246,0.546669
50%,19.000000,0.419198,0.615079,0.378527,39.759220,47.473675,39.912403,0.369220,0.742156,0.274954,0.643988,0.517968,0.691386
75%,28.500000,0.629550,0.709844,0.587565,60.235371,56.205624,55.817419,0.636559,0.857107,0.485970,0.866549,0.639528,0.822013
max,38.000000,0.886700,0.885929,0.865091,107.436958,77.181602,97.667801,0.956261,0.945964,0.847865,1.000000,0.893371,0.989008


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,44.000000,44.000000,42.000000,44.000000,44.000000,43.000000,44.000000,44.000000
mean,21.500000,0.426725,0.525807,0.369579,39.723903,48.289712,44.860008,0.469006,0.659858,0.331832,0.556259,0.471305,0.658620
std,12.845233,0.306738,0.200150,0.277724,28.114911,17.681221,26.754283,0.321917,0.257015,0.284726,0.344086,0.203991,0.252798
min,0.000000,0.000000,0.052160,0.000000,4.000000,4.358899,3.000000,0.000000,0.037431,0.000000,0.000000,0.086001,0.000000
25%,10.750000,0.088582,0.375347,0.106223,13.390266,38.985639,23.975992,0.216573,0.485336,0.057087,0.315271,0.331787,0.555401
50%,21.500000,0.468481,0.547396,0.334102,35.336693,49.220453,45.990274,0.548757,0.742908,0.305611,0.657116,0.485404,0.743848
75%,32.250000,0.663840,0.687700,0.594036,63.085651,57.957726,64.234993,0.739435,0.848911,0.562630,0.839434,0.594763,0.816327
max,43.000000,0.863108,0.849721,0.864922,106.291107,91.766006,97.884621,0.973699,0.968088,0.840022,1.000000,0.886914,0.996899


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,36.000000,39.000000,39.000000,38.000000,39.000000,39.000000,37.000000,39.000000,39.000000
mean,19.000000,0.322056,0.468228,0.356997,39.619508,55.414338,60.800323,0.302170,0.737008,0.348697,0.604810,0.364251,0.525647
std,11.401754,0.339338,0.228671,0.265329,36.317059,21.272043,29.400952,0.325979,0.318879,0.304198,0.429736,0.175956,0.314997
min,0.000000,0.000000,0.000000,0.000000,3.741657,16.093477,6.082763,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.500000,0.001815,0.315214,0.146654,9.933124,42.766579,42.939613,0.003302,0.662509,0.083675,0.014019,0.229551,0.308638
50%,19.000000,0.216511,0.546564,0.308053,30.854423,52.048054,62.968246,0.210127,0.908392,0.249156,0.878650,0.412475,0.571982
75%,28.500000,0.577423,0.631967,0.602424,53.159500,60.491379,76.210003,0.606261,0.957445,0.631970,0.961975,0.490514,0.791063
max,38.000000,0.897651,0.764826,0.827717,132.977448,117.226280,136.036758,0.876046,0.999101,0.958151,1.000000,0.625377,0.979173


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,44.000000,44.000000,42.000000,44.000000,44.000000,43.000000,44.000000,44.000000
mean,21.500000,0.426725,0.525807,0.369579,39.723903,48.289712,44.860008,0.469006,0.659858,0.331832,0.556259,0.471305,0.658620
std,12.845233,0.306738,0.200150,0.277724,28.114911,17.681221,26.754283,0.321917,0.257015,0.284726,0.344086,0.203991,0.252798
min,0.000000,0.000000,0.052160,0.000000,4.000000,4.358899,3.000000,0.000000,0.037431,0.000000,0.000000,0.086001,0.000000
25%,10.750000,0.088582,0.375347,0.106223,13.390266,38.985639,23.975992,0.216573,0.485336,0.057087,0.315271,0.331787,0.555401
50%,21.500000,0.468481,0.547396,0.334102,35.336693,49.220453,45.990274,0.548757,0.742908,0.305611,0.657116,0.485404,0.743848
75%,32.250000,0.663840,0.687700,0.594036,63.085651,57.957726,64.234993,0.739435,0.848911,0.562630,0.839434,0.594763,0.816327
max,43.000000,0.863108,0.849721,0.864922,106.291107,91.766006,97.884621,0.973699,0.968088,0.840022,1.000000,0.886914,0.996899


In [ ]:
tta_ent_sup_lr5_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv'

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv'

In [ ]:
tta_ent_sup_lr4_paths =    "/scratch-second/TTA_Augs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr6_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr6_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,15.000000,17.000000,16.000000,38.000000,39.000000,39.000000,15.000000,17.000000,16.000000
mean,19.000000,0.063585,0.149926,0.143806,23.833599,29.911321,24.496619,0.057451,0.131811,0.110964,0.596544,0.643774,0.836875
std,11.401754,0.145002,0.256374,0.262015,19.401745,19.547246,19.204773,0.165991,0.230503,0.211097,0.342630,0.313145,0.258427
min,0.000000,0.000000,0.000000,0.000000,5.123302,5.099020,3.605551,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.500000,0.000000,0.000000,0.000000,11.868098,16.763054,11.750000,0.000000,0.000000,0.000000,0.324864,0.520207,0.822930
50%,19.000000,0.000000,0.000000,0.000000,19.239279,21.000000,18.435925,0.000000,0.000000,0.000000,0.714801,0.818182,0.938911
75%,28.500000,0.028007,0.152634,0.122345,30.008331,46.108566,31.810621,0.014269,0.182423,0.066129,0.864222,0.862540,0.976989
max,38.000000,0.525128,0.867869,0.839296,77.110313,80.551529,75.520859,0.891972,0.873264,0.746731,1.000000,0.935860,1.000000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,19.000000,0.410169,0.532364,0.370785,42.765808,51.721505,46.969735,0.369377,0.753988,0.311686,0.567900,0.434220,0.595418
std,11.401754,0.307857,0.194297,0.279057,29.827694,15.472049,25.491201,0.295212,0.247267,0.268481,0.347503,0.179040,0.260857
min,0.000000,0.000000,0.069047,0.000000,4.086168,6.164414,8.000000,0.000000,0.065198,0.000000,0.000000,0.067028,0.000000
25%,9.500000,0.148283,0.422048,0.080695,12.675951,44.468868,33.877609,0.086873,0.687205,0.044696,0.354960,0.325079,0.470231
50%,19.000000,0.377731,0.584855,0.423329,43.172459,53.319790,48.228622,0.363236,0.871681,0.311078,0.667805,0.463618,0.662329
75%,28.500000,0.671810,0.673033,0.616992,60.084005,60.662317,61.885786,0.628159,0.938834,0.519053,0.855575,0.545034,0.763628
max,38.000000,0.886159,0.816258,0.810157,107.327286,87.023849,100.756134,0.866565,0.980708,0.857610,0.993865,0.843925,0.961651


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,24.000000,29.000000,28.000000,42.000000,44.000000,44.000000,24.000000,29.000000,28.000000
mean,21.500000,0.137766,0.254054,0.214791,24.922313,30.083382,29.821165,0.120581,0.225562,0.172575,0.676373,0.700221,0.752269
std,12.845233,0.202554,0.287027,0.292808,20.833364,15.670039,24.774945,0.217525,0.272644,0.252928,0.323161,0.252933,0.278527
min,0.000000,0.000000,0.000000,0.000000,5.744563,5.000000,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.750000,0.000000,0.000000,0.000000,11.954865,21.000000,14.168420,0.000000,0.000000,0.000000,0.545861,0.582919,0.671955
50%,21.500000,0.001830,0.128528,0.020095,18.590545,27.000000,20.373440,0.000916,0.069185,0.010153,0.784217,0.809137,0.837299
75%,32.250000,0.308808,0.542816,0.397010,26.517834,38.238617,42.594537,0.186404,0.471647,0.292082,0.917938,0.883158,0.943188
max,43.000000,0.658163,0.868313,0.853820,78.390053,73.576820,89.699501,0.910272,0.853959,0.790047,1.000000,0.990469,1.000000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

NameError: name 'pd' is not defined

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,12.000000,14.000000,13.000000,39.000000,41.000000,41.000000,12.000000,14.000000,13.000000
mean,20.000000,0.087257,0.132989,0.111739,23.638699,31.897608,31.232356,0.078906,0.116524,0.084923,0.692363,0.623940,0.744305
std,11.979149,0.200663,0.235474,0.227999,21.771418,20.304216,27.883270,0.199769,0.206052,0.181767,0.274584,0.317922,0.265639
min,0.000000,0.000000,0.000000,0.000000,3.162278,9.433981,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.000000,0.000000,0.000000,0.000000,10.060505,16.220486,12.000000,0.000000,0.000000,0.000000,0.565321,0.462851,0.701592
50%,20.000000,0.000000,0.000000,0.000000,13.641549,27.384348,21.189621,0.000000,0.000000,0.000000,0.797357,0.761749,0.791260
75%,30.000000,0.018463,0.153124,0.020480,34.004745,46.225486,43.462627,0.009331,0.105932,0.010375,0.883088,0.863760,0.942338
max,40.000000,0.718775,0.748789,0.731986,77.399605,79.655510,89.608604,0.795583,0.660589,0.602450,0.935811,0.928010,0.987021


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,20.000000,0.406915,0.534949,0.359984,36.555751,47.543220,42.150966,0.373015,0.751887,0.308079,0.575133,0.439231,0.600166
std,11.979149,0.301323,0.178649,0.276787,26.474480,16.015006,22.957735,0.283300,0.214401,0.269361,0.376947,0.176193,0.267227
min,0.000000,0.000000,0.072739,0.000000,3.741657,6.633250,5.696340,0.000000,0.073453,0.000000,0.000000,0.062226,0.000000
25%,10.000000,0.120204,0.476242,0.082412,11.565445,42.564655,32.878563,0.122263,0.637574,0.045752,0.253697,0.355629,0.484358
50%,20.000000,0.348877,0.541517,0.357642,36.496574,51.478149,42.043423,0.410366,0.815406,0.255484,0.731403,0.410388,0.629870
75%,30.000000,0.684825,0.682433,0.619893,56.081936,58.905018,57.148930,0.575549,0.928962,0.579297,0.903623,0.583292,0.787331
max,40.000000,0.851473,0.832973,0.805891,94.625053,70.021423,93.908463,0.854937,0.963256,0.752302,0.994186,0.758361,1.000000
